In [1]:
import random
from collections import Counter, defaultdict, deque
from functools import cmp_to_key
from itertools import zip_longest

# 1) Arrays and Hashing

**In plain English.** Instead of searching the rest of the array for something, remember
what you have already seen in a dictionary so the search becomes a single lookup.

> **Analogy.** At a party you want to find the person whose birthday matches yours. Asking
> every guest is $O(n)$ per person. Writing every birthday on a whiteboard as people arrive
> lets each newcomer check the board in one glance.

The only real question is *what key do I store so the answer becomes a lookup?*

| Key you store | Example |
|---|---|
| the value itself | Two Sum, Contains Duplicate |
| a canonical form | sorted letters / letter counts for anagrams |
| a count | Ransom Note, Top K Frequent |
| the last index | Contains Duplicate II |
| the array index itself (values in `1..n`) | First Missing Positive |

**Recognition signals**

- "pair / two numbers that sum to", "seen before", "duplicate", "appears more than"
- "group", "anagram", "same pattern", "frequency", "top k"
- unsorted input and you need $O(n)$ — sorting would cost $O(n \log n)$
- "O(1) extra space" with values in `1..n` → use the array itself as the hash map

**Common bugs**

- Storing the current element *before* checking for its partner, so it pairs with itself.
- Using a `list` as a dict key — convert to `tuple` or a string first.
- Forgetting that `Counter` subtraction drops non-positive counts (handy, but surprising).
- Checking a one-way mapping when the question needs a bijection (Isomorphic Strings).

**Don't confuse with**

| Pattern | Use it when |
|---|---|
| Hash map | input is unsorted, you need "have I seen X" in $O(1)$ |
| Two pointers | input is sorted (or you can sort it) and you need pairs — no extra memory |
| Prefix sum + hash map | the thing you look up is a *running sum*, not a value |
| Sliding window | the answer is a *contiguous* block and validity is monotone |

**Learning path.** The lookup questions start with the pure lookup (Two Sum, Contains Duplicate), moves to
canonical keys (anagrams), counting (Top K, Ransom Note), bijections (Isomorphic, Word
Pattern), O(1)-space voting (Majority), and ends with "array as its own hash map" (First
Missing Positive). The in-place group then drops the hash map and learns in-place rewriting with a write
pointer.

## 1.1 Hash map and set lookups

The ladder: plain lookup → canonical key → frequency count → bijection check → voting
(no map at all) → index marking (the array becomes the map).

### 1.1.1 [1. Two Sum](https://leetcode.com/problems/two-sum/) — ★★★ · Easy

**Problem.** Return indices of the two numbers that add up to `target` (exactly one answer).  
**Signal.** Pair with a given sum, unsorted input, indices required → hash map of value → index.  
*New here: The base move of the whole section: turn an inner search loop into a dict lookup.*

**Fine print.** Ask before coding:
- Exactly one valid pair, or could there be zero or several? → Exactly one is guaranteed, so no "not found" handling is needed.
- Can the same element be used twice? → No, the two indices must differ.
- Negatives or duplicates? → Both allowed, which is why sorting plus two pointers would lose the original indices.
- Return indices or values, and in what order? → Indices, in any order.

> 💡 **Key insight.** For each `x` the partner is fixed (`target - x`), so "find a pair" becomes "have I already seen `target - x`?".

**Brute force →** Try every pair `i < j` → O(n²) time, O(1) space. The inner loop searches for one known value; a hash map answers that search in O(1).

**Steps**
1. Keep a map from value to the index where it was seen.
2. For each `x`, compute the partner `target - x`.
3. If the partner is in the map, return its index and the current one.
4. Otherwise store `x` and continue.

**Complexity**

- **Time:** O(n) — one pass, each lookup is O(1) on average.
- **Space:** O(n) — the map can hold every element.

⚠️ **Trap.** Look up *before* storing `x`, or `x` pairs with itself when `target == 2x`.

In [2]:
def two_sum(nums: list[int], target: int) -> list[int]:
    """Indices i < j with nums[i] + nums[j] == target."""
    # value -> index of everything already scanned
    seen: dict[int, int] = {}
    for i, x in enumerate(nums):
        # the partner was seen earlier: answer found
        if target - x in seen:
            return [seen[target - x], i]
        # store after the check so x never pairs with itself
        seen[x] = i
    return []


assert two_sum([2, 7, 11, 15], 9) == [0, 1]
assert two_sum([3, 2, 4], 6) == [1, 2]
assert two_sum([3, 3], 6) == [0, 1]
print(two_sum([2, 7, 11, 15], 9))

[0, 1]


### 1.1.2 [217. Contains Duplicate](https://leetcode.com/problems/contains-duplicate/) — ★★★ · Easy

**Problem.** Return `True` if any value appears at least twice.  
**Signal.** "Seen before?" → set.  
*New here: When you only need *whether* something was seen, a set replaces the map.*

**Fine print.** Ask before coding:
- Empty or single-element array? → `n ≥ 1`; a single element returns `False`.
- Is extra memory allowed? → If not, sort in place and compare neighbours (O(n log n), O(1)).

> 💡 **Key insight.** A value is a duplicate exactly when it is already in the set of values seen so far.

**Brute force →** Compare every pair → O(n²). Sorting then checking neighbours → O(n log n) time, O(1) space. The set gives O(n) by remembering instead of comparing.

**Steps**
1. Start with an empty set.
2. For each value, if it is in the set, return `True`.
3. Otherwise add it.
4. Finishing the loop means no duplicate.

**Complexity**

- **Time:** O(n) — one pass of O(1) set operations.
- **Space:** O(n) — the set may hold all values.

In [3]:
def contains_duplicate(nums: list[int]) -> bool:
    """True if any value repeats."""
    seen: set[int] = set()
    for x in nums:
        # second sighting: duplicate found, stop early
        if x in seen:
            return True
        seen.add(x)
    return False


assert contains_duplicate([1, 2, 3, 1])
assert not contains_duplicate([1, 2, 3, 4])
print(contains_duplicate([1, 1, 1, 3, 3, 4, 3, 2, 4, 2]))

True


### 1.1.3 [242. Valid Anagram](https://leetcode.com/problems/valid-anagram/) — ★★★ · Easy

**Problem.** Is `t` a rearrangement of `s`?  
**Signal.** Same multiset of characters → compare counts.  
*New here: The value stored is a *count*, not an index.*

**Fine print.** Ask before coding:
- Only lowercase `a–z`, or Unicode? → Lowercase, so a 26-slot array works; Unicode needs a dict/`Counter`.
- Case-sensitive, and do spaces count? → Yes and yes; every character counts as-is.

> 💡 **Key insight.** Two strings are anagrams iff every character occurs the same number of times in both.

**Brute force →** Sort both strings and compare → O(n log n). Counting is O(n) because it never orders anything.

**Steps**
1. If the lengths differ, return `False`.
2. Count the characters of each string.
3. Return whether the two count tables are equal.

**Complexity**

- **Time:** O(n) — one pass per string.
- **Space:** O(1) — at most 26 lowercase keys.

In [4]:
def is_anagram(s: str, t: str) -> bool:
    """True if t is a permutation of s."""
    # different lengths can never match; otherwise compare letter counts
    return len(s) == len(t) and Counter(s) == Counter(t)


assert is_anagram("anagram", "nagaram")
assert not is_anagram("rat", "car")
print(is_anagram("listen", "silent"))

True


### 1.1.4 [49. Group Anagrams](https://leetcode.com/problems/group-anagrams/) — ★★★ · Medium

**Problem.** Group the strings that are anagrams of each other.  
**Signal.** Grouping by an equivalence → hash map from a canonical key to a list.  
*New here: The key is a *canonical form* — every member of a group maps to the same key.*

**Fine print.** Ask before coding:
- Only lowercase letters? → Yes, so the 26-count tuple key works.
- Can a string be empty? → Yes, `""` forms its own group.
- Does the output order matter? → No, any order of groups and of words inside a group.

> 💡 **Key insight.** All anagrams share the same 26-letter count vector, so that vector (as a tuple) is the group's name.

**Brute force →** Compare each word with every group using Valid Anagram → O(n² · k). One canonical key per word avoids all pairwise comparisons.

**Steps**
1. For each word, build its 26-letter count.
2. Turn the count into a tuple so it can be a dict key.
3. Append the word to the list stored under that key.
4. Return all the lists.

**Complexity**

- **Time:** O(n · k) — each of n words of length k is counted once (sorted keys would cost O(n · k log k)).
- **Space:** O(n · k) — every word is stored once.

⚠️ **Trap.** Lists are not hashable — convert the count array to a `tuple`.

In [5]:
def group_anagrams(strs: list[str]) -> list[list[str]]:
    """Buckets of mutually anagrammatic words."""
    groups: dict[tuple[int, ...], list[str]] = defaultdict(list)
    for word in strs:
        # letter-count vector: identical for all anagrams
        count = [0] * 26
        for ch in word:
            count[ord(ch) - ord("a")] += 1
        # tuple makes the vector hashable
        groups[tuple(count)].append(word)
    return list(groups.values())


out = group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"])
assert sorted(map(sorted, out)) == [["ate", "eat", "tea"], ["bat"], ["nat", "tan"]]
print(out)

[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


### 1.1.5 [347. Top K Frequent Elements](https://leetcode.com/problems/top-k-frequent-elements/) — ★★★ · Medium

**Problem.** Return the `k` most frequent values.  
**Signal.** "Top k" by frequency → count, then bucket sort (or heap).  
*New here: Counting plus *bucket sort*: when the sort key is bounded by n, you can index by it.*

**Fine print.** Ask before coding:
- Is the answer unique (no ties at the k-th place)? → Yes, guaranteed.
- Is `k` always valid? → `1 ≤ k ≤` number of distinct values.
- Does the output need to be sorted by frequency? → No, any order.

> 💡 **Key insight.** A frequency is at most `n`, so values can be dropped into bucket `freq` and read from the top bucket down — no comparison sort needed.

**Brute force →** Count, then sort by frequency → O(n log n). A heap of size k gives O(n log k). Buckets give O(n).

**Steps**
1. Count each value.
2. Make `n + 1` buckets and put each value in the bucket of its frequency.
3. Walk buckets from high frequency to low, collecting values.
4. Stop as soon as `k` values are collected.

**Complexity**

- **Time:** O(n) — counting and bucket filling are linear, and there are n + 1 buckets.
- **Space:** O(n) — counts and buckets.

In [6]:
def top_k_frequent(nums: list[int], k: int) -> list[int]:
    """k values with the highest counts."""
    # buckets[f] holds every value seen exactly f times
    buckets: list[list[int]] = [[] for _ in range(len(nums) + 1)]
    for value, freq in Counter(nums).items():
        buckets[freq].append(value)
    # read from the most frequent bucket down
    out: list[int] = []
    for freq in range(len(buckets) - 1, 0, -1):
        out.extend(buckets[freq])
        if len(out) >= k:
            return out[:k]
    return out


assert sorted(top_k_frequent([1, 1, 1, 2, 2, 3], 2)) == [1, 2]
assert top_k_frequent([1], 1) == [1]
print(top_k_frequent([4, 4, 4, 5, 5, 6], 2))

[4, 5]


### 1.1.6 [238. Product of Array Except Self](https://leetcode.com/problems/product-of-array-except-self/) — ★★★ · Medium

**Problem.** `out[i]` = product of every element except `nums[i]`, without division, in O(n).  
**Signal.** "Everything except i" → prefix product × suffix product.  
*New here: Precomputed left/right aggregates — the seed of the prefix-sum idea.*

**Fine print.** Ask before coding:
- Can there be zeros? → Yes, which is exactly why division is banned.
- Does the output array count as extra space? → No, so O(1) extra is the target.
- Can products overflow? → No, each prefix/suffix product fits in a 32-bit int.

> 💡 **Key insight.** Everything except `i` is "everything to the left" times "everything to the right", and both can be built in one sweep each.

**Brute force →** For each `i`, multiply all other elements → O(n²). Division by `nums[i]` breaks on zeros. Prefix and suffix products reuse work instead of recomputing it.

**Steps**
1. Left-to-right pass: `out[i]` = product of all elements before `i`.
2. Right-to-left pass with a running suffix product.
3. Multiply each `out[i]` by the suffix, then fold `nums[i]` into the suffix.

**Complexity**

- **Time:** O(n) — two linear passes.
- **Space:** O(1) extra — the output array holds the prefixes and the suffix is one variable.

In [7]:
def product_except_self(nums: list[int]) -> list[int]:
    """Product of all other elements for each index."""
    n = len(nums)
    # out[i] = product of everything left of i
    out = [1] * n
    for i in range(1, n):
        out[i] = out[i - 1] * nums[i - 1]
    # multiply in the product of everything right of i
    suffix = 1
    for i in range(n - 1, -1, -1):
        out[i] *= suffix
        suffix *= nums[i]
    return out


assert product_except_self([1, 2, 3, 4]) == [24, 12, 8, 6]
assert product_except_self([-1, 1, 0, -3, 3]) == [0, 0, 9, 0, 0]
print(product_except_self([1, 2, 3, 4]))

[24, 12, 8, 6]


### 1.1.7 [128. Longest Consecutive Sequence](https://leetcode.com/problems/longest-consecutive-sequence/) — ★★★ · Medium

**Problem.** Length of the longest run of consecutive integers (any order), in O(n).  
**Signal.** O(n) forbids sorting → set + only start counting at sequence starts.  
*New here: Using the set to decide *where to start* work, so the total work stays linear.*

**Fine print.** Ask before coding:
- Duplicates? → Yes; they must not extend a run, so use a set.
- Empty array? → Return 0.
- Negative numbers? → Yes; only consecutive integer values matter, not positions.

> 💡 **Key insight.** Only count from numbers whose predecessor `x - 1` is missing; each run is then walked exactly once.

**Brute force →** Sort and scan → O(n log n). Walking up from *every* number → O(n²) on one long run. Starting only at run heads makes each number visited at most twice.

**Steps**
1. Put all numbers in a set.
2. For each `x` in the set, skip it if `x - 1` is present (not a run start).
3. From a start, count `x + 1, x + 2, …` while they are in the set.
4. Keep the longest count.

**Complexity**

- **Time:** O(n) — every number is touched once by the outer loop and at most once by a walk.
- **Space:** O(n) — the set.

In [8]:
def longest_consecutive(nums: list[int]) -> int:
    """Length of the longest consecutive-integer run."""
    values = set(nums)
    best = 0
    for x in values:
        # only run heads start a count
        if x - 1 not in values:
            length = 1
            # walk upward while the next integer exists
            while x + length in values:
                length += 1
            best = max(best, length)
    return best


assert longest_consecutive([100, 4, 200, 1, 3, 2]) == 4
assert longest_consecutive([0, 3, 7, 2, 5, 8, 4, 6, 0, 1]) == 9
assert longest_consecutive([]) == 0
print(longest_consecutive([100, 4, 200, 1, 3, 2]))

4


### 1.1.8 [271. Encode and Decode Strings](https://leetcode.com/problems/encode-and-decode-strings/) — ★★★ · Medium · Premium

**Problem.** Serialise a list of strings into one string and back; strings may contain any character.  
**Signal.** Arbitrary content → length-prefix framing (a delimiter alone can be ambiguous).  
*New here: Design of a reversible format — no hashing, but the same "make the lookup unambiguous" thinking.*

**Fine print.** Ask before coding:
- Which characters can appear? → Any of the 256 ASCII characters, including your delimiter, so plain joining is unsafe.
- Can strings be empty, or the list empty? → Both, and decoding must distinguish `[]` from `[""]`.
- Must the encoding be stateless? → Yes, `decode(encode(x))` must work without extra state.

> 💡 **Key insight.** If each string is preceded by its length, the decoder never has to search the content, so any character is safe.

**Brute force →** Join with a delimiter like `","` → breaks as soon as a string contains the delimiter. Escaping works but is fiddly; a length header removes the ambiguity entirely.

**Steps**
1. Encode each string as `len#text` and concatenate.
2. To decode, read digits up to the next `#` to get the length.
3. Take exactly that many characters after `#` as one string.
4. Jump past them and repeat.

**Complexity**

- **Time:** O(total length) — every character is written and read once.
- **Space:** O(total length) — same as time.

In [9]:
def encode(strs: list[str]) -> str:
    """Length-prefixed concatenation."""
    # each piece is "<length>#<text>"
    return "".join(f"{len(s)}#{s}" for s in strs)


def decode(data: str) -> list[str]:
    """Inverse of encode."""
    out: list[str] = []
    i = 0
    while i < len(data):
        # the header ends at the first '#' after i
        j = data.index("#", i)
        length = int(data[i:j])
        # take exactly `length` chars; '#' inside them is harmless
        out.append(data[j + 1 : j + 1 + length])
        i = j + 1 + length
    return out


for case in [["lint", "code", "love", "you"], ["", "a#b", "12#"], []]:
    assert decode(encode(case)) == case
print(encode(["a#b", ""]))

3#a#b0#


### 1.1.9 [36. Valid Sudoku](https://leetcode.com/problems/valid-sudoku/) — ★★ · Medium

**Problem.** Check a partially filled 9×9 board: no repeated digit in any row, column or 3×3 box.  
**Signal.** Several "no duplicates within group" constraints → one set per group.  
*New here: Several sets at once, and mapping a cell to its group (`r // 3, c // 3`).*

**Fine print.** Ask before coding:
- Must the board be solvable? → No, only check that the filled cells follow the rules.
- How are empty cells shown? → `"."`, which must be skipped.

> 💡 **Key insight.** Each filled cell belongs to exactly one row, one column and one box; a repeat in any of those three sets means invalid.

**Brute force →** For each of the 27 groups, collect and check duplicates separately → still O(81) but three passes. One pass with 27 sets checks all constraints together.

**Steps**
1. Create 9 row sets, 9 column sets, 9 box sets.
2. For every filled cell, compute its box index `(r // 3) * 3 + c // 3`.
3. If the digit is already in its row, column or box set, return `False`.
4. Otherwise add it to all three.

**Complexity**

- **Time:** O(81) = O(1) — the board size is fixed.
- **Space:** O(81) = O(1) — same as time.

In [10]:
def is_valid_sudoku(board: list[list[str]]) -> bool:
    """True if no row/column/box repeats a digit."""
    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    boxes = [set() for _ in range(9)]
    for r in range(9):
        for c in range(9):
            d = board[r][c]
            if d == ".":
                continue
            # flatten the 3x3 box coordinate into 0..8
            b = (r // 3) * 3 + c // 3
            # any group already holding d breaks the rules
            if d in rows[r] or d in cols[c] or d in boxes[b]:
                return False
            rows[r].add(d)
            cols[c].add(d)
            boxes[b].add(d)
    return True


board = [
    list("53..7...."),
    list("6..195..."),
    list(".98....6."),
    list("8...6...3"),
    list("4..8.3..1"),
    list("7...2...6"),
    list(".6....28."),
    list("...419..5"),
    list("....8..79"),
]
assert is_valid_sudoku(board)
board[0][0] = "8"
assert not is_valid_sudoku(board)
print("ok")

ok


### 1.1.10 [383. Ransom Note](https://leetcode.com/problems/ransom-note/) — ★★ · Easy

**Problem.** Can `ransom_note` be built from the letters of `magazine` (each letter used once)?  
**Signal.** Multiset containment → counts.  
*New here: Containment (≤) of counts instead of equality.*

**Fine print.** Ask before coding:
- Only lowercase letters? → Yes, so a 26-count array works.
- Can a magazine letter be reused? → No, each letter is used at most once.

> 💡 **Key insight.** The note is buildable iff, for every letter, the note needs no more copies than the magazine has.

**Brute force →** For each note letter, search and delete it from a list of magazine letters → O(m · n). Counting both once is linear.

**Steps**
1. Count the magazine letters.
2. Count the note letters.
3. Subtract: any letter left with a positive count is missing.

**Complexity**

- **Time:** O(m + n) — one count per string.
- **Space:** O(1) — at most 26 keys.

In [11]:
def can_construct(ransom_note: str, magazine: str) -> bool:
    """True if note letters are a sub-multiset of magazine letters."""
    # Counter subtraction keeps only letters the magazine is short of
    return not Counter(ransom_note) - Counter(magazine)


assert not can_construct("a", "b")
assert not can_construct("aa", "ab")
assert can_construct("aa", "aab")
print(can_construct("aa", "aab"))

True


### 1.1.11 [205. Isomorphic Strings](https://leetcode.com/problems/isomorphic-strings/) — ★★ · Easy

**Problem.** Can characters of `s` be replaced to get `t` with a one-to-one mapping?  
**Signal.** Bijection between two alphabets → two maps (or compare first-occurrence patterns).  
*New here: Checking a *bijection* — both directions must be consistent.*

**Fine print.** Ask before coding:
- Are `s` and `t` the same length? → Yes, guaranteed.
- Can a character map to itself? → Yes.
- Can two characters map to the same one? → No, the mapping must be one-to-one both ways.
- Which characters? → Any valid ASCII, so use dicts, not a 26-slot array.

> 💡 **Key insight.** Two strings are isomorphic iff they have the same "shape": replacing each character with the index of its first occurrence gives the same sequence.

**Brute force →** Try every mapping → exponential. Two dictionaries (s→t and t→s) check consistency in one pass; the first-occurrence trick is the same idea in one line.

**Steps**
1. Replace each character of `s` with the index of its first occurrence in `s`.
2. Do the same for `t`.
3. The strings are isomorphic iff the two sequences are equal.

**Complexity**

- **Time:** O(n²) as written (`str.index` scans), fine for short strings; the two-map version is O(n).
- **Space:** O(n) — for the sequences.

⚠️ **Trap.** Checking only `s → t` accepts `"badc" → "baba"`; the reverse map catches it.

In [12]:
def is_isomorphic(s: str, t: str) -> bool:
    """True if a bijective char mapping turns s into t."""
    # same first-occurrence pattern <=> same shape <=> bijection exists
    return [s.index(c) for c in s] == [t.index(c) for c in t]


assert is_isomorphic("egg", "add")
assert not is_isomorphic("foo", "bar")
assert not is_isomorphic("badc", "baba")
print(is_isomorphic("paper", "title"))

True


### 1.1.12 [290. Word Pattern](https://leetcode.com/problems/word-pattern/) — ★★ · Easy

**Problem.** Does the word list follow the letter `pattern` bijectively (`"abba"` ↔ `"dog cat cat dog"`)?  
**Signal.** Same bijection test as Isomorphic Strings, with words instead of characters.  
*New here: A counting shortcut for bijection: distinct letters = distinct words = distinct pairs.*

**Fine print.** Ask before coding:
- Can the number of words differ from the pattern length? → Yes, return `False` in that case.
- Are words separated by exactly one space? → Yes, no leading or trailing spaces.
- Must the mapping be one-to-one? → Yes, two letters cannot share a word.

> 💡 **Key insight.** The mapping is one-to-one iff the number of distinct (letter, word) pairs equals both the number of distinct letters and the number of distinct words.

**Brute force →** Two dictionaries with conflict checks → already O(n); the set-size version is shorter and has the same cost.

**Steps**
1. Split `s` into words; lengths must match the pattern.
2. Count distinct letters, distinct words and distinct (letter, word) pairs.
3. All three counts equal ⇔ bijection.

**Complexity**

- **Time:** O(n) — building three sets.
- **Space:** O(n) — the sets.

In [13]:
def word_pattern(pattern: str, s: str) -> bool:
    """True if words map one-to-one onto pattern letters."""
    words = s.split()
    if len(words) != len(pattern):
        return False
    # an extra pair means some letter or word maps two ways
    return len(set(pattern)) == len(set(words)) == len(set(zip(pattern, words)))


assert word_pattern("abba", "dog cat cat dog")
assert not word_pattern("abba", "dog cat cat fish")
assert not word_pattern("abba", "dog dog dog dog")
print(word_pattern("abba", "dog cat cat dog"))

True


### 1.1.13 [219. Contains Duplicate II](https://leetcode.com/problems/contains-duplicate-ii/) — ★★ · Easy

**Problem.** Are there `i != j` with `nums[i] == nums[j]` and `|i - j| <= k`?  
**Signal.** Duplicate within distance k → map value → last index (or a size-k window set).  
*New here: The stored value is the *latest index*, which bounds distance.*

**Fine print.** Ask before coding:
- Is it `|i - j| <= k` or `< k`? → `<= k`.
- Can `k` be 0? → Yes, then the answer is always `False`.

> 💡 **Key insight.** When a value reappears, the closest earlier copy is the most recent one, so only that index needs to be remembered.

**Brute force →** For each `i`, check the next `k` elements → O(n · k). Storing the last index answers the check in O(1).

**Steps**
1. Keep a map value → most recent index.
2. For each `i, x`, if `x` was seen and `i - last[x] <= k`, return `True`.
3. Update `last[x] = i`.

**Complexity**

- **Time:** O(n) — one pass.
- **Space:** O(n) — the map (a size-k sliding set gives O(k)).

In [14]:
def contains_nearby_duplicate(nums: list[int], k: int) -> bool:
    """True if equal values occur within k positions."""
    # value -> most recent index (the closest candidate)
    last: dict[int, int] = {}
    for i, x in enumerate(nums):
        if x in last and i - last[x] <= k:
            return True
        last[x] = i
    return False


assert contains_nearby_duplicate([1, 2, 3, 1], 3)
assert contains_nearby_duplicate([1, 0, 1, 1], 1)
assert not contains_nearby_duplicate([1, 2, 3, 1, 2, 3], 2)
print("ok")

ok


### 1.1.14 [169. Majority Element](https://leetcode.com/problems/majority-element/) — ★★ · Easy

**Problem.** Return the element that appears more than ⌊n/2⌋ times (guaranteed to exist).  
**Signal.** Strict majority + O(1) space → Boyer–Moore voting.  
*New here: Dropping the hash map entirely: cancellation replaces counting.*

**Fine print.** Ask before coding:
- Is a majority guaranteed? → Yes; if not, Boyer–Moore needs a second counting pass.
- Strictly more than ⌊n/2⌋? → Yes, strictly more.

> 💡 **Key insight.** Cancel each majority copy against a different value; the majority has more copies than everything else combined, so it is the survivor.

**Brute force →** Count with a hash map → O(n) time, O(n) space. Sort and take the middle → O(n log n). Voting keeps O(n) time with O(1) space.

**Steps**
1. Keep a `candidate` and a `count`.
2. When `count` is 0, adopt the current value as the candidate.
3. Same value → `count += 1`; different value → `count -= 1`.
4. The final candidate is the majority.

**Complexity**

- **Time:** O(n) — one pass.
- **Space:** O(1) — two variables.

In [15]:
def majority_element(nums: list[int]) -> int:
    """Value appearing more than n // 2 times."""
    candidate, count = 0, 0
    for x in nums:
        # all votes cancelled: start a new candidate
        if count == 0:
            candidate = x
        # a match adds a vote, a mismatch cancels one
        count += 1 if x == candidate else -1
    return candidate


assert majority_element([3, 2, 3]) == 3
assert majority_element([2, 2, 1, 1, 1, 2, 2]) == 2
print(majority_element([2, 2, 1, 1, 1, 2, 2]))

2


### 1.1.15 [229. Majority Element II](https://leetcode.com/problems/majority-element-ii/) — ★ · Medium

**Problem.** Return all values appearing more than ⌊n/3⌋ times.  
**Signal.** Generalised Boyer–Moore: at most k−1 elements can exceed n/k.  
*New here: Voting with two candidates, and the need to *verify* survivors.*

**Fine print.** Ask before coding:
- Strictly more than ⌊n/3⌋? → Yes, so there are at most two answers.
- Could there be none? → Yes, return `[]`, which is why candidates must be recounted.

> 💡 **Key insight.** At most two values can exceed n/3; cancelling triples of distinct values never removes all copies of such a value.

**Brute force →** Hash-map counts → O(n) space. Two-candidate voting gets O(1) space.

**Steps**
1. Keep two candidates with counts.
2. A match increments its count; an empty slot adopts the value.
3. A value matching neither, with both slots full, decrements both (cancels a triple).
4. Recount the survivors and keep those above n/3.

**Complexity**

- **Time:** O(n) — one voting pass plus one verification pass.
- **Space:** O(1).

⚠️ **Trap.** Survivors are only candidates — without the recount `[1, 2, 3]` would wrongly return values.

In [16]:
def majority_element_ii(nums: list[int]) -> list[int]:
    """All values with count > n // 3."""
    c1, c2, n1, n2 = None, None, 0, 0
    for x in nums:
        # vote for a matching candidate
        if x == c1:
            n1 += 1
        elif x == c2:
            n2 += 1
        # fill an empty slot
        elif n1 == 0:
            c1, n1 = x, 1
        elif n2 == 0:
            c2, n2 = x, 1
        # three distinct values: cancel one of each
        else:
            n1 -= 1
            n2 -= 1
    # survivors are only candidates: verify their real counts
    return [c for c in {c1, c2} if c is not None and nums.count(c) > len(nums) // 3]


assert majority_element_ii([3, 2, 3]) == [3]
assert sorted(majority_element_ii([1, 2])) == [1, 2]
assert majority_element_ii([1]) == [1]
print(majority_element_ii([1, 1, 1, 3, 3, 2, 2, 2]))

[1, 2]


### 1.1.16 [409. Longest Palindrome](https://leetcode.com/problems/longest-palindrome/) — ★★ · Easy

**Problem.** Length of the longest palindrome that can be *built* from the letters of `s`.  
**Signal.** Palindrome from a multiset → pairs plus at most one odd centre.  
*New here: Reasoning about counts' parity instead of the counts themselves.*

**Fine print.** Ask before coding:
- Case-sensitive? → Yes, `"A"` and `"a"` are different letters.
- Do you return the palindrome or its length? → Only the length.

> 💡 **Key insight.** A palindrome is pairs mirrored around at most one single middle letter.

**Brute force →** Generate arrangements and test them → factorial. Counting pairs gives the answer directly.

**Steps**
1. Count each letter.
2. Add `count // 2 * 2` for every letter (all usable pairs).
3. If any count is odd, add 1 for the centre.

**Complexity**

- **Time:** O(n) — one count.
- **Space:** O(1) — bounded alphabet.

In [17]:
def longest_palindrome(s: str) -> int:
    """Max palindrome length buildable from s's letters."""
    counts = Counter(s).values()
    # every pair can sit symmetrically on both sides
    paired = sum(c // 2 * 2 for c in counts)
    # one leftover letter may sit in the middle
    return paired + (1 if any(c % 2 for c in counts) else 0)


assert longest_palindrome("abccccdd") == 7
assert longest_palindrome("a") == 1
print(longest_palindrome("abccccdd"))

7


### 1.1.17 [13. Roman to Integer](https://leetcode.com/problems/roman-to-integer/) — ★★ · Easy

**Problem.** Convert a Roman numeral to an integer.  
**Signal.** Subtractive notation → compare each symbol with the next one.  
*New here: A lookup table plus a one-step look-ahead.*

**Fine print.** Ask before coding:
- Is the input always a valid numeral? → Yes, in range 1–3999.
- Only standard subtractive pairs (IV, IX, XL, XC, CD, CM)? → Yes.

> 💡 **Key insight.** A symbol is subtracted exactly when a larger symbol follows it.

**Brute force →** Special-case all six subtractive pairs (`IV`, `IX`, …) with string matching → works but error-prone. The look-ahead rule covers all of them.

**Steps**
1. Map each symbol to its value.
2. Scan left to right.
3. If the next symbol is larger, subtract this one; otherwise add it.

**Complexity**

- **Time:** O(n) — one pass.
- **Space:** O(1) — the fixed table.

In [18]:
ROMAN = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}


def roman_to_int(s: str) -> int:
    """Integer value of a Roman numeral."""
    total = 0
    for i, ch in enumerate(s):
        # a smaller symbol before a larger one is subtracted (IV, XC, CM)
        if i + 1 < len(s) and ROMAN[ch] < ROMAN[s[i + 1]]:
            total -= ROMAN[ch]
        else:
            total += ROMAN[ch]
    return total


assert roman_to_int("III") == 3
assert roman_to_int("LVIII") == 58
assert roman_to_int("MCMXCIV") == 1994
print(roman_to_int("MCMXCIV"))

1994


### 1.1.18 [12. Integer to Roman](https://leetcode.com/problems/integer-to-roman/) — ★★ · Medium

**Problem.** Convert an integer (1–3999) to a Roman numeral.  
**Signal.** Greedy over a fixed denomination table that already includes the subtractive pairs.  
*New here: Putting the special cases *into the table* so the loop has none.*

**Fine print.** Ask before coding:
- What is the range? → 1–3999, so `M` repeats at most 3 times.
- Standard subtractive form only? → Yes, `4` is `IV`, never `IIII`.

> 💡 **Key insight.** With `900, 400, 90, 40, 9, 4` added as their own symbols, "take the largest value that fits" always produces the correct numeral.

**Brute force →** Handle each decimal digit with if/else for 4 and 9 → long and fragile. The extended table makes it a plain greedy loop.

**Steps**
1. List the 13 values from 1000 down to 1, including the subtractive pairs.
2. For each value, take as many copies as fit (`divmod`).
3. Append that many symbols and continue with the remainder.

**Complexity**

- **Time:** O(1) — the input is bounded by 3999 and the table has 13 entries.
- **Space:** O(1) — same as time.

In [19]:
ROMAN_TABLE = [
    (1000, "M"),
    (900, "CM"),
    (500, "D"),
    (400, "CD"),
    (100, "C"),
    (90, "XC"),
    (50, "L"),
    (40, "XL"),
    (10, "X"),
    (9, "IX"),
    (5, "V"),
    (4, "IV"),
    (1, "I"),
]


def int_to_roman(num: int) -> str:
    """Roman numeral for num."""
    out = []
    for value, symbol in ROMAN_TABLE:
        # how many times this value fits, and what is left
        count, num = divmod(num, value)
        out.append(symbol * count)
    return "".join(out)


assert int_to_roman(3749) == "MMMDCCXLIX"
assert int_to_roman(58) == "LVIII"
assert int_to_roman(1994) == "MCMXCIV"
print(int_to_roman(1994))

MCMXCIV


### 1.1.19 [274. H-Index](https://leetcode.com/problems/h-index/) — ★★ · Medium

**Problem.** Largest `h` such that the researcher has at least `h` papers with ≥ `h` citations each.  
**Signal.** Threshold on counts bounded by n → counting sort (or sort descending).  
*New here: Capping values at n so a counting sort fits in O(n) buckets.*

**Fine print.** Ask before coding:
- Can citation counts be 0 or larger than `n`? → Yes; counts above `n` can be capped at `n`.
- Is the input sorted? → No (H-Index II is the sorted, binary-search version).

> 💡 **Key insight.** `h` can never exceed n, so any citation count above n behaves exactly like n.

**Brute force →** Sort descending and find the last `i` with `citations[i] >= i + 1` → O(n log n). Buckets 0..n give O(n).

**Steps**
1. Bucket each citation count into `min(c, n)`.
2. Walk `h` from n down to 0, accumulating how many papers have ≥ h citations.
3. Return the first `h` where that running count reaches `h`.

**Complexity**

- **Time:** O(n) — one bucketing pass and one walk over n + 1 buckets.
- **Space:** O(n) — the buckets.

In [20]:
def h_index(citations: list[int]) -> int:
    """Researcher's h-index."""
    n = len(citations)
    # counts above n act like n, so n + 1 buckets suffice
    buckets = [0] * (n + 1)
    for c in citations:
        buckets[min(c, n)] += 1
    # papers with at least h citations, accumulated from the top
    at_least = 0
    for h in range(n, -1, -1):
        at_least += buckets[h]
        if at_least >= h:
            return h
    return 0


assert h_index([3, 0, 6, 1, 5]) == 3
assert h_index([1, 3, 1]) == 1
assert h_index([0]) == 0
print(h_index([3, 0, 6, 1, 5]))

3


### 1.1.20 [41. First Missing Positive](https://leetcode.com/problems/first-missing-positive/) — ★★ · Hard

**Problem.** Smallest missing positive integer, O(n) time and O(1) extra space.  
**Signal.** Answer lies in `1..n+1` and values can index the array → cyclic sort / index marking.  
*New here: The array *is* the hash map: value `v` lives at index `v - 1`.*

**Fine print.** Ask before coding:
- Can the array have negatives, zeros or duplicates? → Yes, all three.
- Can I modify the input? → Yes; that is the only way to get O(1) extra space.
- What is the largest possible answer? → `n + 1`, when `1..n` are all present.

> 💡 **Key insight.** With n slots the answer is in `1..n+1`, so putting each value `v` at index `v - 1` and scanning for the first mismatch finds it.

**Brute force →** A set of all values and test 1, 2, 3, … → O(n) time but O(n) space. Sorting → O(n log n). Cyclic sort reaches O(n) time and O(1) space.

**Steps**
1. For each index, while its value `v` is in `1..n` and not already at `v - 1`, swap it there.
2. Ignore values outside `1..n` — they cannot be the answer.
3. Scan: the first index `i` with `nums[i] != i + 1` gives `i + 1`.
4. If every slot matches, the answer is `n + 1`.

**Complexity**

- **Time:** O(n) — each swap puts one value in its final slot, so at most n swaps in total.
- **Space:** O(1) — in place.

⚠️ **Trap.** The loop condition must check `nums[v-1] != v`, otherwise duplicates swap forever.

In [21]:
def first_missing_positive(nums: list[int]) -> int:
    """Smallest positive integer not in nums."""
    n = len(nums)
    for i in range(n):
        # send each in-range value to its home slot v - 1
        while 1 <= nums[i] <= n and nums[nums[i] - 1] != nums[i]:
            j = nums[i] - 1
            nums[i], nums[j] = nums[j], nums[i]
    # the first slot not holding its own value is the gap
    for i in range(n):
        if nums[i] != i + 1:
            return i + 1
    return n + 1


assert first_missing_positive([1, 2, 0]) == 3
assert first_missing_positive([3, 4, -1, 1]) == 2
assert first_missing_positive([7, 8, 9, 11, 12]) == 1
assert first_missing_positive([1, 1]) == 2
print(first_missing_positive([3, 4, -1, 1]))

2


### 1.1.21 [448. Find All Numbers Disappeared in an Array](https://leetcode.com/problems/find-all-numbers-disappeared-in-an-array/) — ★ · Easy

**Problem.** Values are in `1..n`; return the ones that never appear, with O(1) extra space.  
**Signal.** Values index the array → mark presence by negating `nums[v-1]`.  
*New here: Marking with the *sign bit* instead of swapping.*

**Fine print.** Ask before coding:
- Are values guaranteed in `1..n`? → Yes; that is what makes index-marking possible.
- Can I modify the input? → Yes; the output list does not count as extra space.

> 💡 **Key insight.** The sign of `nums[v - 1]` can record "v was seen" without losing the stored value (its absolute value).

**Brute force →** A set of seen values → O(n) space. Negation marking stores the same information inside the array.

**Steps**
1. For each value, take `abs(v)` (it may already be negated).
2. Make `nums[abs(v) - 1]` negative.
3. Indices still holding positive numbers are the missing values (index + 1).

**Complexity**

- **Time:** O(n) — two passes.
- **Space:** O(1) extra — the output list does not count.

In [22]:
def find_disappeared_numbers(nums: list[int]) -> list[int]:
    """Values in 1..n absent from nums."""
    for v in nums:
        # abs: this slot may already have been negated
        idx = abs(v) - 1
        # negative sign at idx records "idx + 1 was seen"
        nums[idx] = -abs(nums[idx])
    # still positive means that value never appeared
    return [i + 1 for i, v in enumerate(nums) if v > 0]


assert find_disappeared_numbers([4, 3, 2, 7, 8, 2, 3, 1]) == [5, 6]
assert find_disappeared_numbers([1, 1]) == [2]
print(find_disappeared_numbers([4, 3, 2, 7, 8, 2, 3, 1]))

[5, 6]


### 1.1.22 [454. 4Sum II](https://leetcode.com/problems/4sum-ii/) — ★ · Medium

**Problem.** Count tuples `(i, j, k, l)` with `a[i] + b[j] + c[k] + d[l] == 0` (four arrays of length n).  
**Signal.** Meet in the middle — split four loops into two halves of two.  
*New here: Meet in the middle: Two Sum where each "value" is itself a pair sum.*

**Fine print.** Ask before coding:
- Do you count index tuples or distinct value tuples? → Index tuples, so duplicate values count separately.
- Are all four arrays the same length? → Yes, `n ≤ 200`, so O(n²) is fine.

> 💡 **Key insight.** `a + b + c + d = 0` ⇔ `a + b = -(c + d)`, so count all `a + b` sums and look up `-(c + d)`.

**Brute force →** Four nested loops → O(n⁴). Storing half the sums turns it into O(n²).

**Steps**
1. Count every pair sum `a[i] + b[j]` in a hash map.
2. For every pair `c[k], d[l]`, add the count of `-(c[k] + d[l])`.
3. Return the total.

**Complexity**

- **Time:** O(n²) — two double loops.
- **Space:** O(n²) — up to n² distinct pair sums.

In [23]:
def four_sum_count(a: list[int], b: list[int], c: list[int], d: list[int]) -> int:
    """Number of zero-sum quadruples across four arrays."""
    # first half: how often each a + b sum occurs
    ab = Counter(x + y for x in a for y in b)
    # second half: each c + d needs a partner of -(c + d)
    return sum(ab[-(x + y)] for x in c for y in d)


assert four_sum_count([1, 2], [-2, -1], [-1, 2], [0, 2]) == 2
assert four_sum_count([0], [0], [0], [0]) == 1
print(four_sum_count([1, 2], [-2, -1], [-1, 2], [0, 2]))

2


### 1.1.23 [554. Brick Wall](https://leetcode.com/problems/brick-wall/) — ★ · Medium

**Problem.** Draw a vertical line through a brick wall crossing the fewest bricks; return that count.  
**Signal.** Maximise edges aligned at the same x → count prefix-sum positions.  
*New here: Counting *prefix sums* (edge positions) — a bridge to the prefix-sum section.*

**Fine print.** Ask before coding:
- Do all rows have the same total width? → Yes.
- Can the line go along the two outer edges? → No, those edges never count.
- How large are widths? → Up to 2³¹, so use a map of edge positions, not an array.

> 💡 **Key insight.** A line crosses a row unless it passes through an edge of that row, so the best x is the most common edge position.

**Brute force →** Try every x from 1 to width − 1 and count crossings → O(width · bricks). Counting edge positions touches each brick once.

**Steps**
1. For each row, walk the bricks keeping a running width.
2. Count each internal edge position (skip the right wall).
3. The answer is `rows − (most common edge count)`.

**Complexity**

- **Time:** O(total bricks) — each brick once.
- **Space:** O(distinct edges) — the counter.

In [24]:
def least_bricks(wall: list[list[int]]) -> int:
    """Minimum bricks crossed by a vertical line."""
    edges: Counter[int] = Counter()
    for row in wall:
        pos = 0
        # internal edges only: the last brick ends at the wall
        for width in row[:-1]:
            pos += width
            edges[pos] += 1
    # rows not having an edge at the best x are crossed
    return len(wall) - max(edges.values(), default=0)


wall = [[1, 2, 2, 1], [3, 1, 2], [1, 3, 2], [2, 4], [3, 1, 2], [1, 3, 1, 1]]
assert least_bricks(wall) == 2
assert least_bricks([[1], [1], [1]]) == 3
print(least_bricks(wall))

2


## 1.2 In-place array manipulation

Many array questions forbid extra memory. The standard tool is a **write pointer**: one index
reads every element, a second index marks where the next kept element goes.

> **Analogy.** Packing a suitcase from a pile: your hand picks up each item (read pointer);
> you place only the keepers at the next free spot (write pointer). Rejected items simply
> get overwritten.

**Recognition signals:** "in place", "O(1) extra memory", "return the new length",
"relative order must be kept", "modify `nums1`".

**Common bugs:** advancing the read pointer after a swap that brought in an unseen value
(Sort Colors); filling a merged array from the front and overwriting unread data (Merge
Sorted Array); forgetting `k %= n` before rotating.

**Learning path:** back-to-front merge → filter (Remove Element) → filter against the last
kept value (Remove Duplicates I, II) → stable partition (Move Zeroes) → reversal tricks
(Rotate) → three-way partition (Sort Colors) → the full pivot/swap/reverse recipe (Next
Permutation).

### 1.2.1 [88. Merge Sorted Array](https://leetcode.com/problems/merge-sorted-array/) — ★★ · Easy

**Problem.** Merge sorted `nums2` into sorted `nums1`, which has `n` spare slots at the end.  
**Signal.** In-place merge with free space at the back → fill from the back.  
*New here: Choosing the *direction* of filling so writes never overwrite unread data.*

**Fine print.** Ask before coding:
- Does `nums1` have exactly `n` spare slots at the end? → Yes, filled with 0s you can overwrite.
- Must I return anything? → No, modify `nums1` in place.
- Can `m` or `n` be 0? → Yes.

> 💡 **Key insight.** The free space is at the back, so placing the largest remaining element there first never destroys anything still to be read.

**Brute force →** Copy `nums2` into the tail and sort → O((m + n) log(m + n)). Merging from the front needs a copy of `nums1` → O(m) extra space. Back-to-front is O(m + n) time and O(1) space.

**Steps**
1. Point `i` at the last real element of `nums1`, `j` at the last of `nums2`, `k` at the last slot.
2. Write the larger of `nums1[i]` and `nums2[j]` at `k` and move that pointer left.
3. Move `k` left after every write.
4. When `nums2` is exhausted, stop — the rest of `nums1` is already in place.

**Complexity**

- **Time:** O(m + n) — each element is written once.
- **Space:** O(1).

In [25]:
def merge(nums1: list[int], m: int, nums2: list[int], n: int) -> None:
    """Merge nums2 into nums1 in place."""
    # i, j: last unread of each list; k: last free slot
    i, j, k = m - 1, n - 1, m + n - 1
    while j >= 0:
        # write the larger tail element into the back slot
        if i >= 0 and nums1[i] > nums2[j]:
            nums1[k] = nums1[i]
            i -= 1
        else:
            nums1[k] = nums2[j]
            j -= 1
        k -= 1


a = [1, 2, 3, 0, 0, 0]
merge(a, 3, [2, 5, 6], 3)
assert a == [1, 2, 2, 3, 5, 6]
b = [0]
merge(b, 0, [1], 1)
assert b == [1]
print(a)

[1, 2, 2, 3, 5, 6]


### 1.2.2 [27. Remove Element](https://leetcode.com/problems/remove-element/) — ★★ · Easy

**Problem.** Remove all occurrences of `val` in place; return the new length.  
**Signal.** Filter in place → write pointer.  
*New here: The basic read/write two-index filter.*

**Fine print.** Ask before coding:
- Must the remaining order be kept? → No, so swapping in from the end is allowed.
- What lies beyond the returned length? → Anything; it is not checked.

> 💡 **Key insight.** Every kept element is copied to the next write slot; the write slot can never overtake the read index, so nothing unread is lost.

**Brute force →** `list.remove` in a loop → O(n²) because each removal shifts the tail. The write pointer moves each element at most once.

**Steps**
1. Start `write = 0`.
2. For each element not equal to `val`, copy it to `nums[write]` and advance `write`.
3. Return `write` (the kept count).

**Complexity**

- **Time:** O(n) — one pass.
- **Space:** O(1).

In [26]:
def remove_element(nums: list[int], val: int) -> int:
    """Compact nums without val; return kept count."""
    write = 0
    for x in nums:
        # keep x by copying it to the next free slot
        if x != val:
            nums[write] = x
            write += 1
    return write


a = [0, 1, 2, 2, 3, 0, 4, 2]
k = remove_element(a, 2)
assert k == 5 and sorted(a[:k]) == [0, 0, 1, 3, 4]
print(k, a[:k])

5 [0, 1, 3, 0, 4]


### 1.2.3 [26. Remove Duplicates from Sorted Array](https://leetcode.com/problems/remove-duplicates-from-sorted-array/) — ★★ · Easy

**Problem.** Remove duplicates in place from a sorted array; return the number of unique values.  
**Signal.** Sorted → duplicates are adjacent → compare with last written value.  
*New here: The keep-condition looks at the *output so far*, not the input.*

**Fine print.** Ask before coding:
- Is the array sorted? → Yes, non-decreasing, so duplicates are adjacent.
- Is extra space allowed? → No, it must be in place.

> 💡 **Key insight.** In a sorted array, `x` is new exactly when it differs from the last value written.

**Brute force →** Build a new list of unique values → O(n) space. Writing in place keeps O(1).

**Steps**
1. Start `write = 0`.
2. Keep `x` if `write == 0` or `x != nums[write - 1]`.
3. Copy and advance on keep; return `write`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [27]:
def remove_duplicates(nums: list[int]) -> int:
    """Unique-compact a sorted array in place."""
    write = 0
    for x in nums:
        # new value iff it differs from the last kept one
        if write == 0 or x != nums[write - 1]:
            nums[write] = x
            write += 1
    return write


a = [0, 0, 1, 1, 1, 2, 2, 3, 3, 4]
k = remove_duplicates(a)
assert a[:k] == [0, 1, 2, 3, 4]
print(k)

5


### 1.2.4 [80. Remove Duplicates from Sorted Array II](https://leetcode.com/problems/remove-duplicates-from-sorted-array-ii/) — ★★ · Medium

**Problem.** Same as above but keep each value at most twice.  
**Signal.** Generalises: allow `x` if it differs from the element two slots back in the output.  
*New here: The "at most k copies" generalisation — compare with `nums[write - k]`.*

**Fine print.** Ask before coding:
- How many copies are kept? → At most 2 (the general version keeps at most `k`).
- Is the array sorted? → Yes.

> 💡 **Key insight.** If `x` equals the value two slots back in the output, two copies are already kept, so this one must go.

**Brute force →** Track a run counter and reset it on value change → works but more state. The look-back of two does it with one comparison.

**Steps**
1. Start `write = 0`.
2. Keep `x` if `write < 2` or `x != nums[write - 2]`.
3. Copy and advance on keep; return `write`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [28]:
def remove_duplicates_ii(nums: list[int]) -> int:
    """Keep at most two copies of each value in a sorted array."""
    write = 0
    for x in nums:
        # equal to the value two slots back means two copies already kept
        if write < 2 or x != nums[write - 2]:
            nums[write] = x
            write += 1
    return write


a = [0, 0, 1, 1, 1, 1, 2, 3, 3]
k = remove_duplicates_ii(a)
assert a[:k] == [0, 0, 1, 1, 2, 3, 3]
print(k)

7


### 1.2.5 [283. Move Zeroes](https://leetcode.com/problems/move-zeroes/) — ★★ · Easy

**Problem.** Move all zeros to the end, keeping the order of non-zeros, in place.  
**Signal.** Stable partition → write pointer with swaps.  
*New here: Swapping instead of copying, so the rejected elements are kept (pushed back).*

**Fine print.** Ask before coding:
- Must the order of non-zero values be kept? → Yes.
- In place with no copy? → Yes; ask whether minimising writes matters as a follow-up.

> 💡 **Key insight.** Swapping each non-zero to the write slot moves the zeros behind it automatically, in one pass.

**Brute force →** Copy non-zeros to a new list and pad with zeros → O(n) space. Swaps keep it O(1).

**Steps**
1. Start `write = 0`.
2. For each non-zero at `i`, swap `nums[i]` with `nums[write]` and advance `write`.
3. Zeros collect after `write`.

**Complexity**

- **Time:** O(n) — one pass.
- **Space:** O(1).

In [29]:
def move_zeroes(nums: list[int]) -> None:
    """Stable move of zeros to the end."""
    write = 0
    for i, x in enumerate(nums):
        # swap the non-zero forward; a zero (if any) drops back to i
        if x != 0:
            nums[write], nums[i] = nums[i], nums[write]
            write += 1


a = [0, 1, 0, 3, 12]
move_zeroes(a)
assert a == [1, 3, 12, 0, 0]
print(a)

[1, 3, 12, 0, 0]


### 1.2.6 [189. Rotate Array](https://leetcode.com/problems/rotate-array/) — ★★ · Medium

**Problem.** Rotate right by `k` steps in place.  
**Signal.** Rotation in place → three reversals.  
*New here: Composing reversals to move whole blocks.*

**Fine print.** Ask before coding:
- Can `k` be larger than `n`? → Yes, so reduce it with `k %= n` first.
- Rotate right or left? → Right.
- Is O(1) extra space required? → Ideally yes, which leads to the triple-reverse trick.

> 💡 **Key insight.** Reversing the whole array puts the last k elements first (but backwards); reversing each block again fixes the order inside it.

**Brute force →** Rotate by one, k times → O(n · k). An extra array → O(n) space. Three reversals are O(n) time and O(1) space.

**Steps**
1. Reduce `k %= n`.
2. Reverse the whole array.
3. Reverse the first `k` elements.
4. Reverse the remaining `n − k` elements.

**Complexity**

- **Time:** O(n) — each element is swapped twice.
- **Space:** O(1).

⚠️ **Trap.** Without `k %= n`, `k > n` indexes past the end.

In [30]:
def rotate(nums: list[int], k: int) -> None:
    """Rotate nums right by k in place."""

    def reverse(lo: int, hi: int) -> None:
        # swap ends inward
        while lo < hi:
            nums[lo], nums[hi] = nums[hi], nums[lo]
            lo, hi = lo + 1, hi - 1

    n = len(nums)
    # rotating by n is a no-op
    k %= n
    # whole reverse brings the last k to the front, backwards
    reverse(0, n - 1)
    # fix the order inside each block
    reverse(0, k - 1)
    reverse(k, n - 1)


a = [1, 2, 3, 4, 5, 6, 7]
rotate(a, 3)
assert a == [5, 6, 7, 1, 2, 3, 4]
b = [-1, -100, 3, 99]
rotate(b, 2)
assert b == [3, 99, -1, -100]
print(a)

[5, 6, 7, 1, 2, 3, 4]


### 1.2.7 [75. Sort Colors](https://leetcode.com/problems/sort-colors/) — ★★ · Medium

**Problem.** Sort an array of 0s, 1s and 2s in one pass, in place.  
**Signal.** Three-way partition → Dutch national flag (three pointers).  
*New here: Three regions maintained by three pointers, with an invariant you can state.*

**Fine print.** Ask before coding:
- Only values 0, 1 and 2? → Yes, so a counting pass also works; one pass needs the Dutch flag.
- Can I use a library sort? → No.

> 💡 **Key insight.** Keep `[0, low)` all 0s, `[low, mid)` all 1s, `(high, end]` all 2s; each step shrinks the unknown region `[mid, high]` by one.

**Brute force →** Count 0s, 1s, 2s and rewrite → two passes (perfectly fine, but not one pass). Comparison sort → O(n log n).

**Steps**
1. Start `low = mid = 0`, `high = n − 1`.
2. `nums[mid] == 0`: swap to `low`, advance `low` and `mid`.
3. `nums[mid] == 1`: advance `mid`.
4. `nums[mid] == 2`: swap to `high`, shrink `high`, and do *not* advance `mid`.
5. Stop when `mid > high`.

**Complexity**

- **Time:** O(n) — each step shrinks the unknown region.
- **Space:** O(1).

⚠️ **Trap.** After swapping with `high`, the new `nums[mid]` is unseen — advancing `mid` skips it.

In [31]:
def sort_colors(nums: list[int]) -> None:
    """In-place one-pass sort of 0/1/2 values."""
    # [0, low) = 0s, [low, mid) = 1s, (high, end] = 2s, [mid, high] unknown
    low, mid, high = 0, 0, len(nums) - 1
    while mid <= high:
        if nums[mid] == 0:
            # grow the 0 region; the swapped-in value is a known 1
            nums[low], nums[mid] = nums[mid], nums[low]
            low += 1
            mid += 1
        elif nums[mid] == 1:
            mid += 1
        else:
            # grow the 2 region; the swapped-in value is unseen, so mid stays
            nums[mid], nums[high] = nums[high], nums[mid]
            high -= 1


a = [2, 0, 2, 1, 1, 0]
sort_colors(a)
assert a == [0, 0, 1, 1, 2, 2]
print(a)

[0, 0, 1, 1, 2, 2]


### 1.2.8 [31. Next Permutation](https://leetcode.com/problems/next-permutation/) — ★★ · Medium

**Problem.** Rearrange into the next lexicographically larger permutation (wrap to smallest), in place.  
**Signal.** Classic three-step algorithm: find pivot, swap with successor, reverse suffix.  
*New here: Combining a scan, a swap and a reversal (from Rotate Array) into one recipe.*

**Fine print.** Ask before coding:
- Duplicates? → Yes, so comparisons must be strict (`<`) in the right places.
- If this is the largest permutation, what happens? → Wrap around to the smallest (sorted ascending).
- In place with O(1) extra space? → Yes.

> 💡 **Key insight.** The longest non-increasing suffix is already at its maximum, so the change must happen just before it — at the pivot — and the suffix must then be reset to its minimum.

**Brute force →** Generate all permutations in order and pick the next → O(n!). The pivot recipe is O(n).

**Steps**
1. From the right, find the first `i` with `nums[i] < nums[i + 1]` (the pivot).
2. If there is a pivot, find the rightmost `j` with `nums[j] > nums[i]` and swap them.
3. Reverse the suffix after `i` so it becomes ascending (smallest).
4. No pivot means the array was the last permutation: reversing everything wraps to the first.

**Complexity**

- **Time:** O(n) — at most three linear scans.
- **Space:** O(1).

In [32]:
def next_permutation(nums: list[int]) -> None:
    """Advance nums to its next permutation in place."""
    # pivot: first i from the right with nums[i] < nums[i + 1]
    i = len(nums) - 2
    while i >= 0 and nums[i] >= nums[i + 1]:
        i -= 1
    if i >= 0:
        # rightmost value larger than the pivot is its smallest successor
        j = len(nums) - 1
        while nums[j] <= nums[i]:
            j -= 1
        nums[i], nums[j] = nums[j], nums[i]
    # the suffix is descending: reverse it to its smallest order
    nums[i + 1 :] = reversed(nums[i + 1 :])


for before, after in [
    ([1, 2, 3], [1, 3, 2]),
    ([3, 2, 1], [1, 2, 3]),
    ([1, 1, 5], [1, 5, 1]),
]:
    next_permutation(before)
    assert before == after
a = [1, 3, 5, 4, 2]
next_permutation(a)
print(a)

[1, 4, 2, 3, 5]


---
# 2) Two Pointers

**In plain English.** Two indices walk through the data and each step permanently rules out
one element, so a double loop collapses into a single pass.

> **Analogy.** Two people search a sorted bookshelf for two books whose page counts add up
> to 500: one starts at the thin end, one at the thick end. Too few pages? The thin-end
> person steps right. Too many? The thick-end person steps left. They never walk back.

| Shape | Pointers | Why it is safe to discard |
|---|---|---|
| Opposite ends | `lo = 0`, `hi = n-1`, move inward | sorted order (or symmetry) proves one end cannot be in any better answer |
| Same direction | `slow`, `fast` | `slow` marks the processed prefix; `fast` reads ahead |

**Recognition signals**

- "sorted array" + "pair / triplet / closest sum"
- "palindrome", "reverse", "in place", "O(1) extra space"
- "subsequence", "merge two sorted …", "compare two strings while skipping"
- maximise/minimise something over a pair `(i, j)` where moving the worse side is provably safe

**Common bugs**

- `while lo <= hi` vs `while lo < hi`: pairs need two distinct elements (`<`); boats allow
  one person alone (`<=`).
- Skipping duplicates only at one level in 3Sum/4Sum → repeated triplets.
- Forgetting to sort first — the discard argument needs sorted order.
- Moving the taller line in Container With Most Water (the area can never improve).

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Hash map (Two Sum) | unsorted input, O(n) space; two pointers needs sorted input but O(1) space |
| Sliding window | also two same-direction pointers, but the *window contents* are the answer and a validity condition drives the left pointer |
| Binary search | discards *half* per step on a sorted range; two pointers discards *one* element per step |
| Slow/fast linked-list pointers | move at different *speeds* to find middles and cycles, not to discard elements |

**Learning path.** Palindrome (pure symmetry) → Two Sum II (the discard proof) → 3Sum (fix
one + Two Sum II + dedupe) → Container / Trapping Rain Water (move the limiting side) →
variants (one deletion, squares, closest, k-sum, greedy pairing). Then the same-direction group switches to
same-direction pointers for subsequences and string scanning, ending with KMP.

## 2.1 Opposite-end pointers

Every question here uses the same loop `while lo < hi:` and differs only in the rule that
decides which pointer moves. State that rule — and why it is safe — before coding.

### 2.1.1 [125. Valid Palindrome](https://leetcode.com/problems/valid-palindrome/) — ★★★ · Easy

**Problem.** After lower-casing and removing non-alphanumerics, is `s` a palindrome?  
**Signal.** Palindrome check → pointers from both ends.  
*New here: The basic inward walk, with skipping of irrelevant characters.*

**Fine print.** Ask before coding:
- What counts as alphanumeric? → Letters and digits; everything else is ignored.
- Case-insensitive? → Yes.
- Is an empty string after cleaning a palindrome? → Yes, `True`.

> 💡 **Key insight.** A palindrome mirrors around its centre, so comparing the outermost relevant characters and moving inward checks everything once.

**Brute force →** Build the cleaned string and compare it with its reverse → O(n) time but O(n) space. Pointers skip junk on the fly with O(1) space.

**Steps**
1. Put `lo` at the start and `hi` at the end.
2. Skip non-alphanumeric characters on either side.
3. Compare the lower-cased characters; a mismatch means `False`.
4. Move both inward until they meet.

**Complexity**

- **Time:** O(n) — each index moves at most n times in total.
- **Space:** O(1).

In [33]:
def is_palindrome(s: str) -> bool:
    """Alphanumeric, case-insensitive palindrome test."""
    lo, hi = 0, len(s) - 1
    while lo < hi:
        # skip punctuation and spaces on either side
        if not s[lo].isalnum():
            lo += 1
        elif not s[hi].isalnum():
            hi -= 1
        # mirrored characters must match
        elif s[lo].lower() != s[hi].lower():
            return False
        else:
            lo, hi = lo + 1, hi - 1
    return True


assert is_palindrome("A man, a plan, a canal: Panama")
assert not is_palindrome("race a car")
assert is_palindrome(" ")
print(is_palindrome("A man, a plan, a canal: Panama"))

True


### 2.1.2 [167. Two Sum II - Input Array Is Sorted](https://leetcode.com/problems/two-sum-ii-input-array-is-sorted/) — ★★ · Medium

**Problem.** Sorted array; return 1-based indices of two numbers summing to `target`, O(1) space.  
**Signal.** Sorted + pair sum → opposite pointers.  
*New here: The discard proof that makes opposite pointers correct.*

**Fine print.** Ask before coding:
- Sorted ascending, exactly one answer? → Yes and yes.
- 1-based indices? → Yes, add 1 to both.
- Can the same element be used twice? → No.

> 💡 **Key insight.** If `a[lo] + a[hi]` is too small, `a[lo]` is too small even with the largest partner, so it can be thrown away; symmetrically for too large.

**Brute force →** All pairs → O(n²). Hash map → O(n) time but O(n) space. Two pointers: O(n) time, O(1) space.

**Steps**
1. `lo` at the start, `hi` at the end.
2. Sum equals target → return.
3. Sum too small → `lo += 1`; too large → `hi -= 1`.
4. Repeat while `lo < hi`.

**Complexity**

- **Time:** O(n) — every step removes one element from consideration.
- **Space:** O(1).

In [34]:
def two_sum_sorted(numbers: list[int], target: int) -> list[int]:
    """1-based indices of the pair summing to target."""
    lo, hi = 0, len(numbers) - 1
    while lo < hi:
        total = numbers[lo] + numbers[hi]
        if total == target:
            return [lo + 1, hi + 1]
        # too small: numbers[lo] fails even with the largest partner
        if total < target:
            lo += 1
        # too large: numbers[hi] fails even with the smallest partner
        else:
            hi -= 1
    return []


assert two_sum_sorted([2, 7, 11, 15], 9) == [1, 2]
assert two_sum_sorted([2, 3, 4], 6) == [1, 3]
assert two_sum_sorted([-1, 0], -1) == [1, 2]
print(two_sum_sorted([2, 7, 11, 15], 9))

[1, 2]


### 2.1.3 [15. 3Sum](https://leetcode.com/problems/3sum/) — ★★★ · Medium

**Problem.** All unique triplets summing to 0.  
**Signal.** Triplet sum → sort, fix one, two-pointer the rest.  
*New here: Reducing k-sum to (k−1)-sum, plus duplicate skipping.*

**Fine print.** Ask before coding:
- Must triplets be unique by value? → Yes, `[-1, 0, 1]` must not appear twice.
- Does the order within a triplet or of the output matter? → No.
- Can the input have duplicates? → Yes, which is why skipping is needed.

> 💡 **Key insight.** Fix the smallest element `nums[i]`; the other two form a Two Sum II with target `-nums[i]` on the sorted suffix.

**Brute force →** Three nested loops plus a set of sorted triplets → O(n³). Sorting + two pointers → O(n²).

**Steps**
1. Sort the array.
2. For each `i`, skip it if it equals the previous value; stop once `nums[i] > 0`.
3. Run two pointers on `i+1 .. n−1` with target `-nums[i]`.
4. On a hit, record it, move both pointers, and skip equal values on the left.

**Complexity**

- **Time:** O(n²) — n outer choices × O(n) pointer scan (sorting O(n log n) is smaller).
- **Space:** O(1) extra — besides the output and the sort.

⚠️ **Trap.** Duplicates must be skipped at *both* levels to keep triplets unique.

In [35]:
def three_sum(nums: list[int]) -> list[list[int]]:
    """Unique triplets with zero sum."""
    nums = sorted(nums)
    out: list[list[int]] = []
    for i in range(len(nums) - 2):
        # smallest element positive: no zero sum possible
        if nums[i] > 0:
            break
        # same first value gives the same triplets
        if i > 0 and nums[i] == nums[i - 1]:
            continue
        # Two Sum II on the suffix
        lo, hi = i + 1, len(nums) - 1
        while lo < hi:
            total = nums[i] + nums[lo] + nums[hi]
            if total < 0:
                lo += 1
            elif total > 0:
                hi -= 1
            else:
                out.append([nums[i], nums[lo], nums[hi]])
                lo += 1
                hi -= 1
                # skip repeated second values
                while lo < hi and nums[lo] == nums[lo - 1]:
                    lo += 1
    return out


assert three_sum([-1, 0, 1, 2, -1, -4]) == [[-1, -1, 2], [-1, 0, 1]]
assert three_sum([0, 1, 1]) == []
assert three_sum([0, 0, 0, 0]) == [[0, 0, 0]]
print(three_sum([-1, 0, 1, 2, -1, -4]))

[[-1, -1, 2], [-1, 0, 1]]


### 2.1.4 [11. Container With Most Water](https://leetcode.com/problems/container-with-most-water/) — ★★★ · Medium

**Problem.** Choose two lines maximising `min(h[i], h[j]) * (j - i)`.  
**Signal.** Pair maximisation on an array → opposite pointers, move the limiting side.  
*New here: The discard proof without sorted input — it comes from the area formula.*

**Fine print.** Ask before coding:
- Can heights be 0? → Yes.
- Does the width count indices between the lines? → Yes, width is `j - i`.
- Can the lines tilt or be moved? → No.

> 💡 **Key insight.** The shorter line caps the height; moving the taller line only loses width and can never raise that cap, so the shorter line is the one to discard.

**Brute force →** All pairs → O(n²). The limiting-side rule visits n pairs, never skipping a better one.

**Steps**
1. Start with the widest pair `lo = 0`, `hi = n − 1`.
2. Record the area `min(h[lo], h[hi]) * (hi − lo)`.
3. Move the pointer at the shorter line inward.
4. Repeat until they meet; return the best area.

**Complexity**

- **Time:** O(n) — one inward pass.
- **Space:** O(1).

In [36]:
def max_area(height: list[int]) -> int:
    """Largest water area between two lines."""
    lo, hi, best = 0, len(height) - 1, 0
    while lo < hi:
        # the shorter line sets the water level
        best = max(best, min(height[lo], height[hi]) * (hi - lo))
        # the shorter line can never do better with a narrower partner
        if height[lo] < height[hi]:
            lo += 1
        else:
            hi -= 1
    return best


assert max_area([1, 8, 6, 2, 5, 4, 8, 3, 7]) == 49
assert max_area([1, 1]) == 1
print(max_area([1, 8, 6, 2, 5, 4, 8, 3, 7]))

49


### 2.1.5 [42. Trapping Rain Water](https://leetcode.com/problems/trapping-rain-water/) — ★★ · Hard

**Problem.** Total water trapped between bars of the given heights.  
**Signal.** Water at i = `min(max_left, max_right) − h[i]` → two pointers with running maxima.  
*New here: Carrying running maxima with the pointers; settling one position per step.*

**Fine print.** Ask before coding:
- Are the bar widths all 1? → Yes.
- Can heights be 0? → Yes, and an empty or single bar holds no water.

> 💡 **Key insight.** If `left_max < right_max`, the water above the left pointer is decided by `left_max` alone — the right side is guaranteed to be at least as tall.

**Brute force →** For each bar, scan left and right for maxima → O(n²). Prefix/suffix max arrays → O(n) time, O(n) space. Two pointers → O(n) time, O(1) space.

**Steps**
1. Pointers at both ends; `left_max = right_max = 0`.
2. Move the side with the lower bar (its max is the bottleneck).
3. Update that side's max, add `side_max − height` to the water.
4. Repeat until the pointers meet.

**Complexity**

- **Time:** O(n) — each bar is settled once.
- **Space:** O(1).

In [37]:
def trap(height: list[int]) -> int:
    """Units of rain water trapped."""
    lo, hi = 0, len(height) - 1
    left_max = right_max = water = 0
    while lo < hi:
        # left side is lower: its own max bounds the water at lo
        if height[lo] < height[hi]:
            left_max = max(left_max, height[lo])
            water += left_max - height[lo]
            lo += 1
        # right side is lower: settle hi with right_max
        else:
            right_max = max(right_max, height[hi])
            water += right_max - height[hi]
            hi -= 1
    return water


assert trap([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]) == 6
assert trap([4, 2, 0, 3, 2, 5]) == 9
print(trap([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]))

6


### 2.1.6 [680. Valid Palindrome II](https://leetcode.com/problems/valid-palindrome-ii/) — ★★ · Easy

**Problem.** Can `s` become a palindrome by deleting at most one character?  
**Signal.** Palindrome with one mismatch allowed → on the first mismatch, try both deletions.  
*New here: Branching once on the first mismatch.*

**Fine print.** Ask before coding:
- Is the deletion at most one, or exactly one? → At most one; a string that is already a palindrome is `True`.
- Is the original string cleaned or case-folded? → No, lowercase letters only, used as-is.

> 💡 **Key insight.** Everything before the first mismatch already matches, so only two candidates remain: drop the left character or drop the right one.

**Brute force →** Delete each character and test → O(n²). Branching only at the first mismatch keeps it O(n).

**Steps**
1. Walk inward while characters match.
2. At the first mismatch, test whether `s[lo+1..hi]` is a palindrome.
3. Or whether `s[lo..hi−1]` is.
4. No mismatch at all → `True`.

**Complexity**

- **Time:** O(n) — one walk plus at most two more.
- **Space:** O(1).

In [38]:
def valid_palindrome_ii(s: str) -> bool:
    """True if deleting at most one char yields a palindrome."""

    def is_pal(lo: int, hi: int) -> bool:
        # plain palindrome check on s[lo..hi]
        while lo < hi:
            if s[lo] != s[hi]:
                return False
            lo, hi = lo + 1, hi - 1
        return True

    lo, hi = 0, len(s) - 1
    while lo < hi:
        # first mismatch: spend the single deletion on either side
        if s[lo] != s[hi]:
            return is_pal(lo + 1, hi) or is_pal(lo, hi - 1)
        lo, hi = lo + 1, hi - 1
    return True


assert valid_palindrome_ii("aba")
assert valid_palindrome_ii("abca")
assert not valid_palindrome_ii("abc")
print(valid_palindrome_ii("abca"))

True


### 2.1.7 [977. Squares of a Sorted Array](https://leetcode.com/problems/squares-of-a-sorted-array/) — ★★ · Easy

**Problem.** Sorted array (may include negatives); return sorted squares in O(n).  
**Signal.** Largest squares sit at the two ends → opposite pointers, fill output from the back.  
*New here: Opposite pointers producing output from largest to smallest.*

**Fine print.** Ask before coding:
- Is the input sorted ascending? → Yes, which is what makes two pointers from the ends work.
- Return a new array? → Yes.

> 💡 **Key insight.** After squaring, the largest value is at one of the two ends, so the ends compete for the last output slot.

**Brute force →** Square and sort → O(n log n). The two ends already give the order, so O(n).

**Steps**
1. Pointers at both ends; fill `out` from index n − 1 downward.
2. Put the larger of `nums[lo]²` and `nums[hi]²` in the current slot.
3. Move that pointer inward.

**Complexity**

- **Time:** O(n) — one pass.
- **Space:** O(n) — the output array.

In [39]:
def sorted_squares(nums: list[int]) -> list[int]:
    """Sorted squares of a sorted array."""
    n = len(nums)
    out = [0] * n
    lo, hi = 0, n - 1
    # fill from the largest square down
    for k in range(n - 1, -1, -1):
        # the bigger absolute value wins the current slot
        if abs(nums[lo]) > abs(nums[hi]):
            out[k] = nums[lo] ** 2
            lo += 1
        else:
            out[k] = nums[hi] ** 2
            hi -= 1
    return out


assert sorted_squares([-4, -1, 0, 3, 10]) == [0, 1, 9, 16, 100]
assert sorted_squares([-7, -3, 2, 3, 11]) == [4, 9, 9, 49, 121]
print(sorted_squares([-4, -1, 0, 3, 10]))

[0, 1, 9, 16, 100]


### 2.1.8 [16. 3Sum Closest](https://leetcode.com/problems/3sum-closest/) — ★★ · Medium

**Problem.** Sum of the triplet closest to `target`.  
**Signal.** Same as 3Sum, but track the best distance instead of exact hits.  
*New here: Optimising a distance instead of finding exact matches.*

**Fine print.** Ask before coding:
- Is exactly one closest sum guaranteed? → Yes, there are no ties to break.
- Return the sum or the triplet? → The sum.

> 💡 **Key insight.** Moving the pointers toward `target` is still safe: a smaller sum can only get closer by increasing, a larger one by decreasing.

**Brute force →** All triplets → O(n³). Sort + two pointers → O(n²).

**Steps**
1. Sort; start `best` at any triplet sum.
2. For each `i`, two pointers on the suffix.
3. Update `best` whenever `|total − target|` improves.
4. `total < target` → `lo += 1`; `total > target` → `hi -= 1`; equal → return immediately.

**Complexity**

- **Time:** O(n²) — n × O(n).
- **Space:** O(1) extra.

In [40]:
def three_sum_closest(nums: list[int], target: int) -> int:
    """Triplet sum nearest to target."""
    nums = sorted(nums)
    best = nums[0] + nums[1] + nums[2]
    for i in range(len(nums) - 2):
        lo, hi = i + 1, len(nums) - 1
        while lo < hi:
            total = nums[i] + nums[lo] + nums[hi]
            # keep the closest sum seen so far
            if abs(total - target) < abs(best - target):
                best = total
            # move the sum toward the target
            if total < target:
                lo += 1
            elif total > target:
                hi -= 1
            else:
                return total
    return best


assert three_sum_closest([-1, 2, 1, -4], 1) == 2
assert three_sum_closest([0, 0, 0], 1) == 0
print(three_sum_closest([-1, 2, 1, -4], 1))

2


### 2.1.9 [18. 4Sum](https://leetcode.com/problems/4sum/) — ★ · Medium

**Problem.** All unique quadruplets summing to `target`.  
**Signal.** k-Sum → recursion that peels one element per level down to Two Sum II.  
*New here: One recursive function that solves any k-Sum.*

**Fine print.** Ask before coding:
- Must quadruplets be unique by value? → Yes.
- Can the sum overflow? → It can exceed 32 bits; fine in Python, use `long` elsewhere.
- Distinct indices only? → Yes.

> 💡 **Key insight.** k-Sum = pick a first element + (k−1)-Sum on the rest, bottoming out at the two-pointer 2-Sum.

**Brute force →** Four loops → O(n⁴). Peeling levels down to two pointers → O(n³).

**Steps**
1. Sort.
2. `k_sum(start, k, goal)`: if `k == 2`, run two pointers with duplicate skipping.
3. Otherwise loop over a first element (skip duplicates) and recurse with `k − 1` and `goal − nums[i]`.
4. Prefix each returned tuple with the chosen element.

**Complexity**

- **Time:** O(n^(k−1)) — k − 2 nested choices times an O(n) two-pointer scan, so O(n³) for 4Sum.
- **Space:** O(k) — recursion depth, besides the output.

In [41]:
def four_sum(nums: list[int], target: int) -> list[list[int]]:
    """Unique quadruplets summing to target."""
    nums = sorted(nums)

    def k_sum(start: int, k: int, goal: int) -> list[list[int]]:
        out: list[list[int]] = []
        # base case: two-pointer 2-Sum with duplicate skipping on the left
        if k == 2:
            lo, hi = start, len(nums) - 1
            while lo < hi:
                total = nums[lo] + nums[hi]
                if total < goal or (lo > start and nums[lo] == nums[lo - 1]):
                    lo += 1
                elif total > goal:
                    hi -= 1
                else:
                    out.append([nums[lo], nums[hi]])
                    lo += 1
                    hi -= 1
            return out
        # peel one element and solve (k-1)-Sum on the rest
        for i in range(start, len(nums) - k + 1):
            if i > start and nums[i] == nums[i - 1]:
                continue
            for rest in k_sum(i + 1, k - 1, goal - nums[i]):
                out.append([nums[i]] + rest)
        return out

    return k_sum(0, 4, target)


assert four_sum([1, 0, -1, 0, -2, 2], 0) == [
    [-2, -1, 1, 2],
    [-2, 0, 0, 2],
    [-1, 0, 0, 1],
]
assert four_sum([2, 2, 2, 2, 2], 8) == [[2, 2, 2, 2]]
print(four_sum([1, 0, -1, 0, -2, 2], 0))

[[-2, -1, 1, 2], [-2, 0, 0, 2], [-1, 0, 0, 1]]


### 2.1.10 [344. Reverse String](https://leetcode.com/problems/reverse-string/) — ★ · Easy

**Problem.** Reverse a list of characters in place.  
**Signal.** In-place reversal → swap ends, move inward.  
*New here: The swap loop reused by Rotate Array, Next Permutation and Reverse Words.*

**Fine print.** Ask before coding:
- In place with O(1) extra memory? → Yes.
- Must I return the list? → No, modify it.

> 💡 **Key insight.** Swapping the two ends and moving inward reverses the list in n/2 swaps.

**Brute force →** `s[::-1]` builds a copy → O(n) space. Swapping in place uses O(1).

**Steps**
1. `lo` at the start, `hi` at the end.
2. Swap `s[lo]` and `s[hi]`.
3. Move both inward until they meet.

**Complexity**

- **Time:** O(n) — n/2 swaps.
- **Space:** O(1).

In [42]:
def reverse_string(s: list[str]) -> None:
    """Reverse s in place."""
    lo, hi = 0, len(s) - 1
    while lo < hi:
        # swap the outer pair and close in
        s[lo], s[hi] = s[hi], s[lo]
        lo, hi = lo + 1, hi - 1


chars = list("hello")
reverse_string(chars)
assert chars == list("olleh")
print(chars)

['o', 'l', 'l', 'e', 'h']


### 2.1.11 [881. Boats to Save People](https://leetcode.com/problems/boats-to-save-people/) — ★ · Medium

**Problem.** Each boat holds at most 2 people and weight `limit`; minimum number of boats.  
**Signal.** Pairing heaviest with lightest → sort + opposite pointers (greedy).  
*New here: Two pointers as a greedy pairing, and `lo <= hi` (one person may ride alone).*

**Fine print.** Ask before coding:
- Is each person's weight ≤ `limit`? → Yes, so everyone fits in a boat alone.
- At most two people per boat? → Yes; with more, greedy pairing no longer works.

> 💡 **Key insight.** The heaviest person needs a boat anyway; giving them the lightest partner wastes the least capacity.

**Brute force →** Try all pairings → exponential. Sort + greedy pairing → O(n log n).

**Steps**
1. Sort weights.
2. The heaviest (`hi`) always boards a new boat.
3. If the lightest (`lo`) fits too, board them as well.
4. Count boats until everyone is on.

**Complexity**

- **Time:** O(n log n) — sorting dominates; the pointer pass is O(n).
- **Space:** O(1) extra.

In [43]:
def num_rescue_boats(people: list[int], limit: int) -> int:
    """Fewest two-seat boats needed."""
    people = sorted(people)
    lo, hi, boats = 0, len(people) - 1, 0
    # <= because a single person may still need a boat
    while lo <= hi:
        # the lightest joins the heaviest if they fit together
        if people[lo] + people[hi] <= limit:
            lo += 1
        # the heaviest always leaves on this boat
        hi -= 1
        boats += 1
    return boats


assert num_rescue_boats([1, 2], 3) == 1
assert num_rescue_boats([3, 2, 2, 1], 3) == 3
assert num_rescue_boats([3, 5, 3, 4], 5) == 4
print(num_rescue_boats([3, 2, 2, 1], 3))

3


## 2.2 Same-direction pointers

Both pointers move left to right (or right to left). One pointer usually reads every
character; the other advances only when a condition holds — a match, a surviving character,
or a kept element. Ask: *when does the slow pointer move?*

### 2.2.1 [392. Is Subsequence](https://leetcode.com/problems/is-subsequence/) — ★★ · Easy

**Problem.** Is `s` a subsequence of `t`?  
**Signal.** Match in order, skipping allowed → one pointer per string.  
*New here: Same-direction pointers: one always moves, the other moves on a match.*

**Fine print.** Ask before coding:
- Is an empty `s` a subsequence? → Yes, always `True`.
- Follow-up: many `s` queries on the same `t`? → Precompute positions of each letter in `t` and binary-search.

> 💡 **Key insight.** Matching each character of `s` as early as possible in `t` never hurts, so a greedy scan decides it.

**Brute force →** Try every subset of `t` positions → exponential. Greedy matching is one pass.

**Steps**
1. Pointer `i` into `s` starts at 0.
2. Walk through `t`; when `t`'s character equals `s[i]`, advance `i`.
3. `s` is a subsequence iff `i` reaches `len(s)`.

**Complexity**

- **Time:** O(|t|) — one pass over `t`.
- **Space:** O(1).

In [44]:
def is_subsequence(s: str, t: str) -> bool:
    """True if s can be obtained by deleting chars from t."""
    i = 0
    for ch in t:
        # greedy: consume s[i] at its earliest match in t
        if i < len(s) and s[i] == ch:
            i += 1
    return i == len(s)


assert is_subsequence("abc", "ahbgdc")
assert not is_subsequence("axc", "ahbgdc")
assert is_subsequence("", "abc")
print(is_subsequence("abc", "ahbgdc"))

True


### 2.2.2 [844. Backspace String Compare](https://leetcode.com/problems/backspace-string-compare/) — ★★ · Easy

**Problem.** `#` is a backspace; are `s` and `t` equal after typing? O(1) space.  
**Signal.** Backspace affects characters to the *left* → scan from the right with a skip counter.  
*New here: Scanning backwards because the edit affects what came before.*

**Fine print.** Ask before coding:
- What does `#` on an empty text do? → Nothing; the text stays empty.
- Is O(1) space required? → Ideally, which forces scanning from the right.

> 💡 **Key insight.** Reading from the right, a `#` tells you exactly how many upcoming characters to skip, so survivors can be produced one at a time.

**Brute force →** Simulate typing with a stack for each string → O(m + n) time and space. The backward scan uses O(1) space.

**Steps**
1. Walk each string from the end with a `skip` counter.
2. `#` → `skip += 1`; a character with `skip > 0` → consume one skip.
3. Otherwise the character survives — yield it.
4. Compare the two survivor streams.

**Complexity**

- **Time:** O(m + n) — each character read once.
- **Space:** O(1) extra — for the scan (the `list(...)` here is for brevity; comparing generators pairwise keeps O(1)).

In [45]:
def backspace_compare(s: str, t: str) -> bool:
    """Equality of strings after applying '#' backspaces."""

    def survivors(text: str):
        # pending backspaces seen to the right
        skip = 0
        for ch in reversed(text):
            if ch == "#":
                skip += 1
            # this character is erased by a later '#'
            elif skip:
                skip -= 1
            else:
                yield ch

    return list(survivors(s)) == list(survivors(t))


assert backspace_compare("ab#c", "ad#c")
assert backspace_compare("ab##", "c#d#")
assert not backspace_compare("a#c", "b")
print(backspace_compare("ab#c", "ad#c"))

True


### 2.2.3 [1768. Merge Strings Alternately](https://leetcode.com/problems/merge-strings-alternately/) — ★ · Easy

**Problem.** Merge two strings by alternating characters; append the leftover tail.  
**Signal.** Walk both strings with one index.  
*New here: A shared index for two sequences of different length.*

**Fine print.** Ask before coding:
- Can the strings have different lengths? → Yes, append the leftover tail.
- Which string starts? → `word1`.

> 💡 **Key insight.** Slicing `w[i : i + 1]` returns `""` past the end, so one loop handles the uneven tail.

**Brute force →** Already linear; the only risk is index errors on the longer tail.

**Steps**
1. Loop `i` up to the longer length.
2. Append `word1[i]` if it exists, then `word2[i]` if it exists.
3. Join the pieces.

**Complexity**

- **Time:** O(m + n) — each character once.
- **Space:** O(m + n) — the output string.

In [46]:
def merge_alternately(word1: str, word2: str) -> str:
    """Interleave two strings."""
    out = []
    for i in range(max(len(word1), len(word2))):
        # a slice past the end is "" instead of an IndexError
        out.append(word1[i : i + 1])
        out.append(word2[i : i + 1])
    return "".join(out)


assert merge_alternately("abc", "pqr") == "apbqcr"
assert merge_alternately("ab", "pqrs") == "apbqrs"
print(merge_alternately("abcd", "pq"))

apbqcd


### 2.2.4 [151. Reverse Words in a String](https://leetcode.com/problems/reverse-words-in-a-string/) — ★★ · Medium

**Problem.** Reverse the order of words; collapse extra spaces.  
**Signal.** In Python `split()` handles spaces; in-place version = reverse whole, then each word.  
*New here: Reversal at two levels (whole string, then each word).*

**Fine print.** Ask before coding:
- Can there be leading, trailing or multiple spaces? → Yes; the output has single spaces and no ends.
- Can I use `split()`? → Yes in Python; an in-place O(1) version reverses the whole string, then each word.

> 💡 **Key insight.** Reversing the whole string reverses the word order *and* each word; reversing each word again restores its spelling.

**Brute force →** `split` + reverse + `join` is already O(n) and is the Python answer. With a mutable char array and O(1) space: reverse all, reverse each word, compact spaces with a write pointer.

**Steps**
1. Split on whitespace (drops extra spaces).
2. Reverse the list of words.
3. Join with single spaces.

**Complexity**

- **Time:** O(n) — split, reverse and join are each linear.
- **Space:** O(n) — the word list.

In [47]:
def reverse_words(s: str) -> str:
    """Words of s in reverse order, single-spaced."""
    # split() without arguments also collapses repeated spaces
    return " ".join(reversed(s.split()))


assert reverse_words("the sky is blue") == "blue is sky the"
assert reverse_words("  hello world  ") == "world hello"
assert reverse_words("a good   example") == "example good a"
print(reverse_words("the sky is blue"))

blue is sky the


### 2.2.5 [28. Find the Index of the First Occurrence in a String](https://leetcode.com/problems/find-the-index-of-the-first-occurrence-in-a-string/) — ★★ · Easy

**Problem.** Index of the first occurrence of `needle` in `haystack`, or −1.  
**Signal.** Substring search → sliding comparison (KMP for guaranteed linear time).  
*New here: KMP: a same-direction scan that never moves the text pointer backwards.*

**Fine print.** Ask before coding:
- What if `needle` is empty? → Return 0 (the constraints say `len ≥ 1`, but say what you would do).
- Is a built-in like `find` allowed? → Usually not; the expected answer is a sliding compare or KMP.

> 💡 **Key insight.** After a partial match fails, the longest prefix of the needle that is also a suffix of what matched can be kept — no need to restart.

**Brute force →** Compare the needle at every start → O(m · n) worst case (e.g. `"aaaa…ab"`). KMP is O(m + n).

**Steps**
1. Build `lps[i]`: the longest proper prefix of `needle[:i+1]` that is also its suffix.
2. Scan the haystack with a needle pointer `j`.
3. On a mismatch, fall back `j = lps[j − 1]` (repeat while needed) instead of restarting.
4. On a match, advance `j`; when `j == len(needle)`, return the start index.

**Complexity**

- **Time:** O(m + n) — `j` rises at most once per character and each fallback lowers it, so total work is linear.
- **Space:** O(n) — for `lps` (n = needle length).

In [48]:
def str_str(haystack: str, needle: str) -> int:
    """First index of needle in haystack (KMP)."""
    # lps[i]: longest proper prefix of needle[:i+1] that is also a suffix
    lps = [0] * len(needle)
    j = 0
    for i in range(1, len(needle)):
        while j and needle[i] != needle[j]:
            j = lps[j - 1]
        if needle[i] == needle[j]:
            j += 1
        lps[i] = j
    # scan the haystack, reusing matched prefixes on mismatch
    j = 0
    for i, ch in enumerate(haystack):
        while j and ch != needle[j]:
            j = lps[j - 1]
        if ch == needle[j]:
            j += 1
        # whole needle matched: it started len(needle) - 1 chars back
        if j == len(needle):
            return i - j + 1
    return -1


assert str_str("sadbutsad", "sad") == 0
assert str_str("leetcode", "leeto") == -1
assert str_str("aabaaabaaac", "aabaaac") == 4
print(str_str("hello", "ll"))

2


---
# 3) Sliding Window

**In plain English.** Look at a contiguous block through a window that only ever slides
right. The right edge grows the window; the left edge shrinks it when the window breaks
the rule. Each element enters once and leaves once, so the scan is $O(n)$.

> **Analogy.** A caterpillar crawling along a branch: the head stretches forward to eat
> new leaves; when the body gets too long (the rule breaks), the tail pulls up. It never
> crawls backwards.

```python
left = 0
for right, x in enumerate(arr):
    add(x)                              # head moves forward
    while window_is_invalid():          # tail catches up
        remove(arr[left]); left += 1
    best = max(best, right - left + 1)  # longest valid window
```

For the *shortest* valid window, update the answer *inside* the while-loop instead.

**Recognition signals**

- "longest / shortest **substring** or **subarray** such that …"
- "at most k distinct / at most k flips / at most k replacements"
- "contains all characters of t", "anagram / permutation of p in s"
- "every window of size k" (fixed window)
- all values non-negative (sums only grow when the window grows)

**Common bugs**

- Using a window when values can be negative — the sum is no longer monotone; use prefix
  sums + hash map instead.
- Updating the answer outside the while-loop for "shortest" (or inside it for "longest").
- Moving `left` backwards (`left = last[ch] + 1` without `max`).
- Leaving zero-count keys in the map when the rule counts *distinct* keys.

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Two pointers (opposite ends) | pointers start at both ends; the window starts empty on the left |
| Prefix sum + hash map | handles negative numbers and "exactly k" sums; the window needs monotonicity |
| Monotonic deque | a helper *inside* a window when you need its max/min in O(1) |
| DP (Kadane) | "best subarray sum" with negatives — no fixed validity rule to shrink on |

**The monotonicity test.** A sliding window works only if: once a window is invalid,
every bigger window containing it is invalid too. Say this sentence for the problem before
coding; if it fails, the window is the wrong tool.

**Learning path.** Running minimum (Stock) → longest window with a jump (Longest
Substring) → longest window with a count rule (Character Replacement) → shortest window
(Minimum Window, Min Size Subarray) → "at most k bad" family (Max Ones III, Fruit, k
distinct) → "exactly k = at most k − at most k−1" → inverted problems (Reduce X to Zero).
The fixed-size group then fixes the window size and ends with the monotonic deque.

## 3.1 Variable-size windows

Variable windows: the right edge always advances; the left edge advances only to restore
validity. Before coding, name the *rule* that makes a window valid and the *state* you need
to check it in O(1) (a sum, a count map, a zero counter).

### 3.1.1 [121. Best Time to Buy and Sell Stock](https://leetcode.com/problems/best-time-to-buy-and-sell-stock/) — ★★★ · Easy

**Problem.** One buy then one later sell; maximum profit (0 if none).  
**Signal.** Best pair `i < j` maximising `p[j] − p[i]` → track the running minimum.  
*New here: The degenerate window: only the cheapest point to the left matters.*

**Fine print.** Ask before coding:
- Must the buy happen before the sell? → Yes.
- Exactly one transaction? → At most one; return 0 if prices only fall.

> 💡 **Key insight.** For any sell day, the best buy day is simply the cheapest day before it.

**Brute force →** Try every buy/sell pair → O(n²). Remembering the minimum so far makes each sell day O(1).

**Steps**
1. Keep `min_price` = cheapest price seen so far.
2. For each price, update `min_price`.
3. Profit if sold today = `price − min_price`; keep the best.

**Complexity**

- **Time:** O(n) — one pass.
- **Space:** O(1) — two variables.

In [49]:
def max_profit(prices: list[int]) -> int:
    """Best single-trade profit."""
    min_price, best = float("inf"), 0
    for p in prices:
        # cheapest buy available up to today
        min_price = min(min_price, p)
        # profit if we sell today
        best = max(best, p - min_price)
    return best


assert max_profit([7, 1, 5, 3, 6, 4]) == 5
assert max_profit([7, 6, 4, 3, 1]) == 0
print(max_profit([7, 1, 5, 3, 6, 4]))

5


### 3.1.2 [3. Longest Substring Without Repeating Characters](https://leetcode.com/problems/longest-substring-without-repeating-characters/) — ★★★ · Medium

**Problem.** Length of the longest substring with all distinct characters.  
**Signal.** Longest contiguous block with a property → sliding window.  
*New here: The real window; `left` can *jump* using a last-index map.*

**Fine print.** Ask before coding:
- What character set? → Letters, digits, symbols and spaces, so use a dict, not 26 slots.
- Substring or subsequence? → Substring, so it must be contiguous.
- Can `s` be empty? → Yes, return 0.

> 💡 **Key insight.** When `s[right]` repeats inside the window, every window starting at or before its previous copy is invalid, so `left` jumps just past that copy.

**Brute force →** Check every substring for distinct characters → O(n³), or O(n²) with a set per start. The window visits each index once.

**Steps**
1. Map each character to its last index.
2. For each `right`, if `s[right]` was last seen at or after `left`, move `left` to that index + 1.
3. Update the last index of `s[right]`.
4. Record `right − left + 1`.

**Complexity**

- **Time:** O(n) — `right` moves n times and `left` only moves forward.
- **Space:** O(alphabet) — the last-index map.

⚠️ **Trap.** Only jump `left` forward — use `max(left, last[ch] + 1)` (`"abba"` breaks otherwise).

In [50]:
def length_of_longest_substring(s: str) -> int:
    """Longest substring without repeated characters."""
    # char -> index of its latest occurrence
    last: dict[str, int] = {}
    left = best = 0
    for right, ch in enumerate(s):
        # repeat inside the window: jump left past the old copy (never backwards)
        if ch in last:
            left = max(left, last[ch] + 1)
        last[ch] = right
        best = max(best, right - left + 1)
    return best


assert length_of_longest_substring("abcabcbb") == 3
assert length_of_longest_substring("bbbbb") == 1
assert length_of_longest_substring("pwwkew") == 3
assert length_of_longest_substring("abba") == 2
print(length_of_longest_substring("abcabcbb"))

3


### 3.1.3 [424. Longest Repeating Character Replacement](https://leetcode.com/problems/longest-repeating-character-replacement/) — ★★★ · Medium

**Problem.** Replace at most `k` characters; longest substring of one repeated letter.  
**Signal.** Window is valid while `size − (count of most frequent char) ≤ k`.  
*New here: A validity rule built from a count map, and a window that never shrinks.*

**Fine print.** Ask before coding:
- Only uppercase letters? → Yes, so `max_freq` over 26 counts is cheap.
- Is it at most `k` replacements? → Yes, at most; using fewer is fine.

> 💡 **Key insight.** The cheapest fix keeps the most frequent letter and replaces the rest, so a window is valid iff `size − max_freq ≤ k`.

**Brute force →** For every substring, count letters and test the rule → O(n²·26). The window keeps the counts incrementally.

**Steps**
1. Grow the window and count the new letter.
2. Track `max_freq`, the highest single-letter count seen.
3. If `size − max_freq > k`, slide `left` forward by one (the window keeps its size).
4. The final window size is the answer.

**Complexity**

- **Time:** O(n) — each step does O(1) work.
- **Space:** O(26) — letter counts.

⚠️ **Trap.** `max_freq` is never decreased: a stale value only stops the window growing, it never makes the answer too large.

In [51]:
def character_replacement(s: str, k: int) -> int:
    """Longest same-letter substring after at most k replacements."""
    count: Counter[str] = Counter()
    left = max_freq = 0
    for right, ch in enumerate(s):
        count[ch] += 1
        # best single-letter count ever seen in a window
        max_freq = max(max_freq, count[ch])
        # letters to replace exceed k: slide (window size stays the same)
        if right - left + 1 - max_freq > k:
            count[s[left]] -= 1
            left += 1
    # the window never shrank, so its final size is the best size
    return len(s) - left


assert character_replacement("ABAB", 2) == 4
assert character_replacement("AABABBA", 1) == 4
print(character_replacement("AABABBA", 1))

4


### 3.1.4 [76. Minimum Window Substring](https://leetcode.com/problems/minimum-window-substring/) — ★★★ · Hard

**Problem.** Smallest substring of `s` that contains every character of `t` (with multiplicity).  
**Signal.** Shortest window covering a requirement → expand until valid, shrink while valid.  
*New here: The *shortest* window template, and a single `missing` counter for O(1) validity.*

**Fine print.** Ask before coding:
- Do duplicates in `t` count? → Yes, `"aa"` needs two `a`s in the window.
- Case-sensitive? → Yes, uppercase and lowercase differ.
- No valid window? → Return `""`. Is the answer unique? → Yes.

> 💡 **Key insight.** Expand until the window covers `t`, then shrink from the left as far as it stays covering — every minimal window is found this way.

**Brute force →** Test every substring against `t`'s counts → O(n²·alphabet). The window is O(|s| + |t|).

**Steps**
1. `need` = counts of `t`; `missing` = `len(t)`.
2. Expand `right`: if `need[ch] > 0`, one requirement is met (`missing −= 1`); decrement `need[ch]`.
3. While `missing == 0`, record the window, then return `s[left]` to `need` and move `left`.
4. If returning it makes `need > 0`, the window is short again (`missing += 1`).

**Complexity**

- **Time:** O(|s| + |t|) — each index enters and leaves once.
- **Space:** O(alphabet) — the `need` map.

In [52]:
def min_window(s: str, t: str) -> str:
    """Shortest substring of s containing all of t."""
    # need[c] > 0: still required; < 0: surplus inside the window
    need = Counter(t)
    missing = len(t)
    left = 0
    best = (0, float("inf"))
    for right, ch in enumerate(s):
        # a still-required char reduces the shortfall
        if need[ch] > 0:
            missing -= 1
        need[ch] -= 1
        # window covers t: shrink it as far as possible
        while missing == 0:
            if right - left < best[1] - best[0]:
                best = (left, right)
            need[s[left]] += 1
            # dropped a required char: window no longer covers t
            if need[s[left]] > 0:
                missing += 1
            left += 1
    return "" if best[1] == float("inf") else s[best[0] : best[1] + 1]


assert min_window("ADOBECODEBANC", "ABC") == "BANC"
assert min_window("a", "a") == "a"
assert min_window("a", "aa") == ""
print(min_window("ADOBECODEBANC", "ABC"))

BANC


### 3.1.5 [209. Minimum Size Subarray Sum](https://leetcode.com/problems/minimum-size-subarray-sum/) — ★★ · Medium

**Problem.** Positive integers; length of the shortest subarray with sum ≥ `target` (0 if none).  
**Signal.** Positive values → window sum is monotone → shortest-window template.  
*New here: The shortest-window template on sums.*

**Fine print.** Ask before coding:
- Are all values positive? → Yes; with negatives the sliding window breaks.
- Sum `≥ target` or exactly equal? → `≥ target`.
- No such subarray? → Return 0.

> 💡 **Key insight.** With positive numbers, once a window reaches the target, shrinking from the left is the only way it can get shorter and still qualify.

**Brute force →** Every start, extend until the sum reaches target → O(n²). Prefix sums + binary search → O(n log n). The window is O(n).

**Steps**
1. Add `nums[right]` to the running sum.
2. While the sum ≥ target, record the length and remove `nums[left]`.
3. Return the best length, or 0 if none.

**Complexity**

- **Time:** O(n) — each element is added and removed once.
- **Space:** O(1).

In [53]:
def min_sub_array_len(target: int, nums: list[int]) -> int:
    """Shortest subarray length with sum >= target."""
    left = total = 0
    best = len(nums) + 1
    for right, x in enumerate(nums):
        total += x
        # valid: record, then try a shorter window
        while total >= target:
            best = min(best, right - left + 1)
            total -= nums[left]
            left += 1
    return best if best <= len(nums) else 0


assert min_sub_array_len(7, [2, 3, 1, 2, 4, 3]) == 2
assert min_sub_array_len(4, [1, 4, 4]) == 1
assert min_sub_array_len(11, [1, 1, 1, 1, 1, 1, 1, 1]) == 0
print(min_sub_array_len(7, [2, 3, 1, 2, 4, 3]))

2


### 3.1.6 [1004. Max Consecutive Ones III](https://leetcode.com/problems/max-consecutive-ones-iii/) — ★ · Medium

**Problem.** Binary array; flip at most `k` zeros; longest run of 1s.  
**Signal.** "At most k bad elements in the window" → longest-window template counting zeros.  
*New here: Reframing "flip k" as "window may contain at most k bad elements".*

**Fine print.** Ask before coding:
- Only 0s and 1s? → Yes.
- Can `k` be 0 or larger than the number of zeros? → Yes, both.

> 💡 **Key insight.** Flipping is just a story — the answer is the longest window with at most `k` zeros.

**Brute force →** Every start, extend while zeros ≤ k → O(n²). The window is O(n).

**Steps**
1. Count zeros in the window as `right` advances.
2. While zeros > k, shrink from the left (decrement if a zero leaves).
3. Record the window size.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [54]:
def longest_ones(nums: list[int], k: int) -> int:
    """Longest 1-run after flipping at most k zeros."""
    left = zeros = best = 0
    for right, x in enumerate(nums):
        zeros += x == 0
        # too many zeros to flip: shrink
        while zeros > k:
            zeros -= nums[left] == 0
            left += 1
        best = max(best, right - left + 1)
    return best


assert longest_ones([1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0], 2) == 6
assert longest_ones([0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1], 3) == 10
print(longest_ones([1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0], 2))

6


### 3.1.7 [1493. Longest Subarray of 1's After Deleting One Element](https://leetcode.com/problems/longest-subarray-of-1s-after-deleting-one-element/) — ★ · Medium

**Problem.** Delete exactly one element; longest subarray of only 1s.  
**Signal.** Same as Max Consecutive Ones III with `k = 1`, minus the deleted element.  
*New here: Adjusting the template's answer (size − 1) for a mandatory operation.*

**Fine print.** Ask before coding:
- Must exactly one element be deleted? → Yes, even when the array is all 1s, so `[1,1,1]` gives 2.

> 💡 **Key insight.** The best result is a window with at most one zero, minus one element for the deletion.

**Brute force →** Delete each element and find the longest run → O(n²). The window is O(n).

**Steps**
1. Longest window with at most one zero.
2. Record `right − left` (window size minus the deleted element).
3. All-ones input still deletes one 1, which the formula already handles.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [55]:
def longest_subarray_after_delete(nums: list[int]) -> int:
    """Longest all-ones subarray after deleting one element."""
    left = zeros = best = 0
    for right, x in enumerate(nums):
        zeros += x == 0
        # at most one zero (the one we delete)
        while zeros > 1:
            zeros -= nums[left] == 0
            left += 1
        # size minus the mandatory deletion
        best = max(best, right - left)
    return best


assert longest_subarray_after_delete([1, 1, 0, 1]) == 3
assert longest_subarray_after_delete([0, 1, 1, 1, 0, 1, 1, 0, 1]) == 5
assert longest_subarray_after_delete([1, 1, 1]) == 2
print(longest_subarray_after_delete([0, 1, 1, 1, 0, 1, 1, 0, 1]))

5


### 3.1.8 [904. Fruit Into Baskets](https://leetcode.com/problems/fruit-into-baskets/) — ★ · Medium

**Problem.** Longest subarray containing at most 2 distinct values.  
**Signal.** "At most k distinct" → window with a count map; shrink while map has > k keys.  
*New here: The validity rule is the *number of keys* in the count map.*

**Fine print.** Ask before coding:
- Exactly two baskets, each holding one fruit type? → Yes, so at most 2 distinct values.
- Must you move strictly to the right without skipping trees? → Yes, so it is a contiguous subarray.

> 💡 **Key insight.** "Two baskets" means the window may contain at most two distinct values; `len(count)` measures that if empty keys are deleted.

**Brute force →** Every start, extend while ≤ 2 kinds → O(n²). The window is O(n).

**Steps**
1. Add `fruits[right]` to the count map.
2. While the map has more than two keys, remove `fruits[left]`, deleting keys that hit zero.
3. Record the window size.

**Complexity**

- **Time:** O(n).
- **Space:** O(1) — at most three keys at any time.

⚠️ **Trap.** A key with count 0 still counts in `len(count)` — delete it.

In [56]:
def total_fruit(fruits: list[int]) -> int:
    """Longest subarray with at most two distinct values."""
    count: Counter[int] = Counter()
    left = best = 0
    for right, f in enumerate(fruits):
        count[f] += 1
        # three kinds: drop from the left until one kind disappears
        while len(count) > 2:
            count[fruits[left]] -= 1
            if count[fruits[left]] == 0:
                del count[fruits[left]]
            left += 1
        best = max(best, right - left + 1)
    return best


assert total_fruit([1, 2, 1]) == 3
assert total_fruit([0, 1, 2, 2]) == 3
assert total_fruit([1, 2, 3, 2, 2]) == 4
print(total_fruit([1, 2, 3, 2, 2]))

4


### 3.1.9 [340. Longest Substring with At Most K Distinct Characters](https://leetcode.com/problems/longest-substring-with-at-most-k-distinct-characters/) — ★ · Medium · Premium

**Problem.** Longest substring with at most `k` distinct characters.  
**Signal.** Fruit Into Baskets with a general `k`.  
*New here: Nothing new — confirm the template generalises by changing one constant.*

**Fine print.** Ask before coding:
- Can `k` be 0? → Yes, return 0.
- What character set? → Any, so count with a dict.

> 💡 **Key insight.** Same rule as Fruit Into Baskets: valid while `len(count) <= k`.

**Brute force →** Every substring → O(n²). The window is O(n).

**Steps**
1. Add `s[right]` to the count map.
2. While more than `k` keys, remove `s[left]` and delete zero keys.
3. Record the window size.

**Complexity**

- **Time:** O(n).
- **Space:** O(k) — at most k + 1 keys.

In [57]:
def length_of_longest_substring_k_distinct(s: str, k: int) -> int:
    """Longest substring with at most k distinct characters."""
    count: Counter[str] = Counter()
    left = best = 0
    for right, ch in enumerate(s):
        count[ch] += 1
        # too many distinct characters: shrink
        while len(count) > k:
            count[s[left]] -= 1
            if count[s[left]] == 0:
                del count[s[left]]
            left += 1
        best = max(best, right - left + 1)
    return best


assert length_of_longest_substring_k_distinct("eceba", 2) == 3
assert length_of_longest_substring_k_distinct("aa", 1) == 2
print(length_of_longest_substring_k_distinct("eceba", 2))

3


### 3.1.10 [992. Subarrays with K Different Integers](https://leetcode.com/problems/subarrays-with-k-different-integers/) — ★ · Hard

**Problem.** Count subarrays with exactly `k` distinct values.  
**Signal.** "Exactly k" is not monotone, but "at most k" is → `exact(k) = at_most(k) − at_most(k−1)`.  
*New here: Counting subarrays (not just the longest) and the "exactly = at most − at most" trick.*

**Fine print.** Ask before coding:
- Exactly `k` distinct, not at most? → Exactly, which is why it is computed as `at_most(k) - at_most(k - 1)`.
- Count subarrays, not lengths? → Yes, count them.

> 💡 **Key insight.** Every valid window `[left, right]` contains `right − left + 1` valid subarrays ending at `right`, and "exactly k" is the difference of two "at most" counts.

**Brute force →** Every subarray with a set → O(n²). Two window passes → O(n).

**Steps**
1. Write `at_most(limit)`: the k-distinct window that adds `right − left + 1` per step.
2. Return `at_most(k) − at_most(k − 1)`.

**Complexity**

- **Time:** O(n) — two linear window passes.
- **Space:** O(k) — the count map.

In [58]:
def subarrays_with_k_distinct(nums: list[int], k: int) -> int:
    """Number of subarrays with exactly k distinct values."""

    def at_most(limit: int) -> int:
        count: Counter[int] = Counter()
        left = total = 0
        for right, x in enumerate(nums):
            count[x] += 1
            while len(count) > limit:
                count[nums[left]] -= 1
                if count[nums[left]] == 0:
                    del count[nums[left]]
                left += 1
            # every suffix of the valid window ending at right qualifies
            total += right - left + 1
        return total

    # exactly k = at most k minus at most k - 1
    return at_most(k) - at_most(k - 1)


assert subarrays_with_k_distinct([1, 2, 1, 2, 3], 2) == 7
assert subarrays_with_k_distinct([1, 2, 1, 3, 4], 3) == 3
print(subarrays_with_k_distinct([1, 2, 1, 2, 3], 2))

7


### 3.1.11 [1838. Frequency of the Most Frequent Element](https://leetcode.com/problems/frequency-of-the-most-frequent-element/) — ★ · Medium

**Problem.** Up to `k` increments by 1 in total; maximise the count of one value.  
**Signal.** After sorting, make a window all equal to its largest element; cost is `max × size − sum`.  
*New here: Sorting first so a window has a computable "cost", then sliding on that cost.*

**Fine print.** Ask before coding:
- Can you only increase values? → Yes, only `+1` steps, which is why the array is sorted first.
- Is `k` the total budget across all elements? → Yes.

> 💡 **Key insight.** In sorted order, the cheapest group to equalise is a contiguous window raised to its right end, costing `nums[right] × size − window_sum`.

**Brute force →** For each target value, greedily raise the nearest smaller values → O(n²). Sort + window → O(n log n).

**Steps**
1. Sort.
2. Grow the window, keeping its sum.
3. While `nums[right] × size − sum > k`, shrink from the left.
4. Record the window size.

**Complexity**

- **Time:** O(n log n) — the sort; the window is O(n).
- **Space:** O(1) extra (besides the sorted copy).

In [59]:
def max_frequency(nums: list[int], k: int) -> int:
    """Highest achievable frequency with k unit increments."""
    nums = sorted(nums)
    left = total = best = 0
    for right, x in enumerate(nums):
        total += x
        # cost of raising the whole window to x exceeds the budget
        while x * (right - left + 1) - total > k:
            total -= nums[left]
            left += 1
        best = max(best, right - left + 1)
    return best


assert max_frequency([1, 2, 4], 5) == 3
assert max_frequency([1, 4, 8, 13], 5) == 2
assert max_frequency([3, 9, 6], 2) == 1
print(max_frequency([1, 4, 8, 13], 5))

2


### 3.1.12 [1658. Minimum Operations to Reduce X to Zero](https://leetcode.com/problems/minimum-operations-to-reduce-x-to-zero/) — ★ · Medium

**Problem.** Remove elements from either end so the removed sum is exactly `x`; fewest removals.  
**Signal.** Removing from both ends = keeping a middle block → longest subarray with sum `total − x`.  
*New here: Inverting the problem: "remove from the ends" = "keep a middle window".*

**Fine print.** Ask before coding:
- Are all values positive? → Yes; that is what lets the middle window slide.
- Impossible? → Return −1.
- Must the removed sum be exactly `x`? → Yes.

> 💡 **Key insight.** Whatever is removed from the two ends leaves a contiguous middle summing to `total − x`; fewest removals = longest such middle.

**Brute force →** Try every split of removals between left and right → O(n²). The inverted window is O(n).

**Steps**
1. `goal = sum(nums) − x`; negative goal → impossible.
2. Longest window with sum exactly `goal` (positive values, so shrink while sum > goal).
3. Answer `n − longest`, or −1 if none.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [60]:
def min_operations(nums: list[int], x: int) -> int:
    """Fewest end-removals summing to x."""
    # the kept middle block must sum to this
    goal = sum(nums) - x
    if goal < 0:
        return -1
    left = window = 0
    longest = -1
    for right, v in enumerate(nums):
        window += v
        # positive values: too big can only be fixed by shrinking
        while window > goal:
            window -= nums[left]
            left += 1
        if window == goal:
            longest = max(longest, right - left + 1)
    # fewest removals = everything outside the longest kept block
    return -1 if longest == -1 else len(nums) - longest


assert min_operations([1, 1, 4, 2, 3], 5) == 2
assert min_operations([5, 6, 7, 8, 9], 4) == -1
assert min_operations([3, 2, 20, 1, 1, 3], 10) == 5
print(min_operations([3, 2, 20, 1, 1, 3], 10))

5


## 3.2 Fixed-size windows

For a window of fixed length `k`, add `arr[right]` and remove `arr[right - k]` on every
step after the first `k`. No while-loop is needed — the size is the rule. The recurring
question is *what summary of the window do I keep* (a sum, a count map, a deque).

### 3.2.1 [567. Permutation in String](https://leetcode.com/problems/permutation-in-string/) — ★★ · Medium

**Problem.** Does `s2` contain a permutation of `s1` as a substring?  
**Signal.** Anagram of a fixed-length pattern → fixed window of size `len(s1)` with counts.  
*New here: The fixed window: one letter in, one letter out each step.*

**Fine print.** Ask before coding:
- Only lowercase letters? → Yes, so two 26-count arrays work.
- What if `s1` is longer than `s2`? → Return `False`.

> 💡 **Key insight.** A permutation of `s1` is any window of length `len(s1)` with the same letter counts.

**Brute force →** Sort or count every window from scratch → O(n · k). Sliding updates two counts per step.

**Steps**
1. Count `s1` and the first `k` characters of `s2`.
2. If equal, return `True`.
3. Slide: add `s2[right]`, remove `s2[right − k]`, compare again.

**Complexity**

- **Time:** O(n) — each comparison is over at most 26 keys.
- **Space:** O(26).

In [61]:
def check_inclusion(s1: str, s2: str) -> bool:
    """True if some substring of s2 is an anagram of s1."""
    k = len(s1)
    target = Counter(s1)
    # first window of length k
    window = Counter(s2[:k])
    if window == target:
        return True
    for right in range(k, len(s2)):
        # one char enters, one leaves
        window[s2[right]] += 1
        window[s2[right - k]] -= 1
        if window == target:
            return True
    return False


assert check_inclusion("ab", "eidbaooo")
assert not check_inclusion("ab", "eidboaoo")
print(check_inclusion("ab", "eidbaooo"))

True


### 3.2.2 [438. Find All Anagrams in a String](https://leetcode.com/problems/find-all-anagrams-in-a-string/) — ★★ · Medium

**Problem.** All start indices in `s` of anagrams of `p`.  
**Signal.** Permutation in String, collecting every match instead of stopping.  
*New here: A single loop that both builds and slides the window.*

**Fine print.** Ask before coding:
- Can matches overlap? → Yes, report every start index.
- Only lowercase letters? → Yes.

> 💡 **Key insight.** Same as Permutation in String; every matching window's start is `right − k + 1`.

**Brute force →** Count each window from scratch → O(n · k). Sliding → O(n).

**Steps**
1. Add `s[right]` to the window counts.
2. Once `right ≥ k`, remove `s[right − k]`.
3. If the counts equal `p`'s, record `right − k + 1`.

**Complexity**

- **Time:** O(n).
- **Space:** O(26).

⚠️ **Trap.** `Counter` equality treats a zero count like a missing key (Python 3.10+), which is what we want here.

In [62]:
def find_anagrams(s: str, p: str) -> list[int]:
    """Start indices of p's anagrams in s."""
    k = len(p)
    target, window = Counter(p), Counter()
    out = []
    for right, ch in enumerate(s):
        window[ch] += 1
        # keep the window at length k
        if right >= k:
            window[s[right - k]] -= 1
        if window == target:
            out.append(right - k + 1)
    return out


assert find_anagrams("cbaebabacd", "abc") == [0, 6]
assert find_anagrams("abab", "ab") == [0, 1, 2]
print(find_anagrams("cbaebabacd", "abc"))

[0, 6]


### 3.2.3 [643. Maximum Average Subarray I](https://leetcode.com/problems/maximum-average-subarray-i/) — ★ · Easy

**Problem.** Maximum average of a contiguous subarray of length `k`.  
**Signal.** Fixed window → running sum.  
*New here: The simplest window summary: one running sum.*

**Fine print.** Ask before coding:
- Is `k ≤ n` always? → Yes.
- How precise must the result be? → Within 1e-5, so return a float.

> 💡 **Key insight.** Moving the window by one changes its sum by `new − old`.

**Brute force →** Sum each window → O(n · k). The running sum is O(n).

**Steps**
1. Sum the first `k` elements.
2. Slide: add the entering element, subtract the leaving one.
3. Keep the max sum; divide by `k` at the end.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [63]:
def find_max_average(nums: list[int], k: int) -> float:
    """Best average over windows of length k."""
    window = best = sum(nums[:k])
    for right in range(k, len(nums)):
        # slide by one: add the new element, drop the oldest
        window += nums[right] - nums[right - k]
        best = max(best, window)
    return best / k


assert find_max_average([1, 12, -5, -6, 50, 3], 4) == 12.75
assert find_max_average([5], 1) == 5.0
print(find_max_average([1, 12, -5, -6, 50, 3], 4))

12.75


### 3.2.4 [1423. Maximum Points You Can Obtain from Cards](https://leetcode.com/problems/maximum-points-you-can-obtain-from-cards/) — ★ · Medium

**Problem.** Take exactly `k` cards from the two ends; maximise the sum.  
**Signal.** Taking from ends = leaving a contiguous middle of length `n − k`.  
*New here: The inversion trick again, now with a fixed size.*

**Fine print.** Ask before coding:
- Must you take exactly `k` cards? → Yes.
- Can `k` equal `n`? → Yes, then take everything.

> 💡 **Key insight.** Maximising what you take = minimising the contiguous block of `n − k` cards you leave.

**Brute force →** Try every split `i` left + `k − i` right → O(k²) with re-summing, O(k) with care. The window is O(n) and very simple.

**Steps**
1. Window size = `n − k`.
2. Find the minimum window sum by sliding.
3. Answer = `total − min_window`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [64]:
def max_score(card_points: list[int], k: int) -> int:
    """Best sum of k cards taken from the ends."""
    n = len(card_points)
    # the cards left behind form a middle block of this size
    size = n - k
    window = best = sum(card_points[:size])
    for right in range(size, n):
        window += card_points[right] - card_points[right - size]
        best = min(best, window)
    # take everything except the cheapest middle block
    return sum(card_points) - best


assert max_score([1, 2, 3, 4, 5, 6, 1], 3) == 12
assert max_score([2, 2, 2], 2) == 4
assert max_score([9, 7, 7, 9, 7, 7, 9], 7) == 55
print(max_score([1, 2, 3, 4, 5, 6, 1], 3))

12


### 3.2.5 [239. Sliding Window Maximum](https://leetcode.com/problems/sliding-window-maximum/) — ★★ · Hard

**Problem.** Maximum of every window of size `k`.  
**Signal.** Window max with O(1) amortised updates → monotonic decreasing deque of indices.  
*New here: The monotonic deque — a window summary for max/min.*

**Fine print.** Ask before coding:
- Is `k` always between 1 and `n`? → Yes.
- Duplicates and negatives? → Both allowed, so the deque must handle equal values.

> 💡 **Key insight.** A value with a larger value to its right can never again be a window max, so it can be thrown away immediately.

**Brute force →** Scan each window for its max → O(n · k). A heap → O(n log n). The deque is O(n).

**Steps**
1. Keep a deque of indices whose values decrease front to back.
2. Before pushing `i`, pop back indices with values ≤ `nums[i]`.
3. Pop the front if it has slid out of the window.
4. Once the first window is full, the front is the max.

**Complexity**

- **Time:** O(n) — each index is pushed and popped at most once.
- **Space:** O(k) — the deque.

In [65]:
def max_sliding_window(nums: list[int], k: int) -> list[int]:
    """Maximum of each length-k window."""
    # indices with decreasing values; front = current max
    dq: deque[int] = deque()
    out = []
    for i, x in enumerate(nums):
        # smaller values behind x can never be a max again
        while dq and nums[dq[-1]] <= x:
            dq.pop()
        dq.append(i)
        # front index left the window
        if dq[0] == i - k:
            dq.popleft()
        if i >= k - 1:
            out.append(nums[dq[0]])
    return out


assert max_sliding_window([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7]
assert max_sliding_window([1], 1) == [1]
print(max_sliding_window([1, 3, -1, -3, 5, 3, 6, 7], 3))

[3, 3, 5, 5, 6, 7]


### 3.2.6 [30. Substring with Concatenation of All Words](https://leetcode.com/problems/substring-with-concatenation-of-all-words/) — ★★ · Hard

**Problem.** Start indices where a concatenation of all `words` (equal length `w`, any order) appears.  
**Signal.** Fixed word length → run a word-level sliding window for each of the `w` offsets.  
*New here: A window whose unit is a *word*, run once per alignment offset.*

**Fine print.** Ask before coding:
- Are all words the same length? → Yes; the solution relies on it.
- Can `words` contain duplicates? → Yes, so count them with a `Counter`.
- Must every word be used exactly once? → Yes, in any order.

> 💡 **Key insight.** Every match starts at some offset `0..w−1` and then advances in whole words, so `w` separate word-level windows cover all cases.

**Brute force →** At every start, chop `m` words and compare counts → O(n · m · w). Word windows are O(n · w).

**Steps**
1. For each offset `0..w−1`, step through `s` in chunks of `w`.
2. Unknown word → reset the window just after it.
3. Known word → add it; while it is over-used, drop words from the left.
4. When the window holds exactly `m` words, record `left`.

**Complexity**

- **Time:** O(n · w) — each offset scans n/w chunks of w characters, and there are w offsets.
- **Space:** O(m) — word counts.

In [66]:
def find_substring(s: str, words: list[str]) -> list[int]:
    """Starts of concatenations of all words."""
    w, m = len(words[0]), len(words)
    need = Counter(words)
    out = []
    # one word-level window per alignment
    for offset in range(w):
        left, window, used = offset, Counter(), 0
        for right in range(offset, len(s) - w + 1, w):
            word = s[right : right + w]
            # a foreign word breaks every window containing it
            if word not in need:
                left, window, used = right + w, Counter(), 0
                continue
            window[word] += 1
            used += 1
            # too many copies of word: drop words from the left
            while window[word] > need[word]:
                window[s[left : left + w]] -= 1
                left += w
                used -= 1
            if used == m:
                out.append(left)
    return sorted(out)


assert find_substring("barfoothefoobarman", ["foo", "bar"]) == [0, 9]
assert (
    find_substring("wordgoodgoodgoodbestword", ["word", "good", "best", "word"]) == []
)
assert find_substring("barfoofoobarthefoobarman", ["bar", "foo", "the"]) == [6, 9, 12]
print(find_substring("barfoothefoobarman", ["foo", "bar"]))

[0, 9]


---
# 4) Prefix Sums

**In plain English.** Store running totals once; then the sum of any range is just the
difference of two totals. Paired with a hash map, "does some subarray sum to k?" becomes
"have I seen the running total `current − k` before?".

> **Analogy.** A car's odometer. To know how far you drove between two towns you don't
> re-measure the road — you subtract the two odometer readings.

`prefix[i] = nums[0] + … + nums[i-1]`, so `sum(nums[i:j]) = prefix[j] − prefix[i]`.

**Recognition signals**

- "subarray sum **equals** k", "number of subarrays with sum …", "divisible by k"
- negative numbers present (sliding window fails, prefix sums don't)
- "equal number of 0s and 1s" (map 0 → −1, look for sum 0)
- many range-sum queries on a fixed array
- many "add v to every element in [l, r]" updates → difference array

**Common bugs**

- Forgetting the seed `{0: 1}` (count) or `{0: -1}` (first index) — misses subarrays that
  start at index 0.
- Storing the *latest* index when you want the *longest* subarray (store the first).
- Updating the map before the lookup, so a subarray of length 0 is counted.
- Off-by-one between `prefix` of length `n + 1` and the original array.

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Sliding window | needs non-negative values and a monotone rule; prefix + map works with negatives and "exactly k" |
| Two Sum hash map | same lookup idea, but the stored values are running totals instead of elements |
| Difference array | the inverse: store changes, then a prefix sum rebuilds the values |
| Kadane (DP) | finds the *best* subarray sum, not the *count* of subarrays with a given sum |

**Learning path.** Count subarrays with sum k (the core trick) → longest with sum 0
(store first index) → remainders mod k (count / length ≥ 2) → static range queries in 1D
and 2D → running left/right balance → difference arrays → prefix sums as a CDF for
weighted sampling.

## 4.1 Prefix sum with hash map

The core equation: subarray `(i, j]` sums to `k` ⇔ `prefix[j] − prefix[i] = k` ⇔
`prefix[i] = prefix[j] − k`. So at each `j`, ask the map about `prefix[j] − k`. Store
*counts* when counting subarrays, *first indices* when maximising length.

### 4.1.1 [560. Subarray Sum Equals K](https://leetcode.com/problems/subarray-sum-equals-k/) — ★★ · Medium

**Problem.** Count contiguous subarrays summing to `k` (values may be negative).  
**Signal.** Negative values break sliding windows → prefix sum + hash map of prefix counts.  
*New here: The core trick: Two Sum over running totals.*

**Fine print.** Ask before coding:
- Can values be negative? → Yes, so a sliding window fails and prefix sums plus a hashmap are needed.
- Count subarrays or return them? → Count.
- Non-empty and contiguous? → Yes.

> 💡 **Key insight.** A subarray ending here sums to `k` exactly when an earlier running total equals `current − k`.

**Brute force →** Sum every subarray → O(n²) with running sums. The map of prefix counts answers each end position in O(1).

**Steps**
1. Seed the map with `{0: 1}` (the empty prefix).
2. Add each element to the running prefix.
3. Add `count[prefix − k]` to the answer.
4. Then record the current prefix in the map.

**Complexity**

- **Time:** O(n) — one pass of O(1) map operations.
- **Space:** O(n) — up to n distinct prefixes.

⚠️ **Trap.** Without the `{0: 1}` seed, subarrays starting at index 0 are missed.

In [67]:
def subarray_sum(nums: list[int], k: int) -> int:
    """Count of subarrays with sum k."""
    # prefix value -> how many times it occurred; 0 = empty prefix
    seen = Counter({0: 1})
    prefix = total = 0
    for x in nums:
        prefix += x
        # each earlier prefix equal to prefix - k starts a valid subarray
        total += seen[prefix - k]
        # record after the lookup so empty subarrays are not counted
        seen[prefix] += 1
    return total


assert subarray_sum([1, 1, 1], 2) == 2
assert subarray_sum([1, 2, 3], 3) == 2
assert subarray_sum([1, -1, 0], 0) == 3
print(subarray_sum([1, 1, 1], 2))

2


### 4.1.2 [525. Contiguous Array](https://leetcode.com/problems/contiguous-array/) — ★★ · Medium

**Problem.** Binary array; longest subarray with equal numbers of 0s and 1s.  
**Signal.** Map 0 → −1; equal counts ⇔ subarray sum 0 ⇔ two equal prefix sums.  
*New here: Recoding values so the condition becomes "sum = 0", and storing *first* indices for length.*

**Fine print.** Ask before coding:
- Only 0s and 1s? → Yes, which makes the "treat 0 as −1" trick possible.
- Return the length, not the subarray? → Length.

> 💡 **Key insight.** With 0 as −1, a balanced subarray sums to 0, which means the running total repeats; the farthest repeat gives the longest.

**Brute force →** Count zeros and ones for every subarray → O(n²). First-index map → O(n).

**Steps**
1. Running total: +1 for a 1, −1 for a 0.
2. Seed `first = {0: -1}`.
3. If the total was seen before, the gap `i − first[total]` is balanced; keep the max.
4. Otherwise store `i` as the first index of this total.

**Complexity**

- **Time:** O(n).
- **Space:** O(n) — the first-index map.

In [68]:
def find_max_length(nums: list[int]) -> int:
    """Longest balanced 0/1 subarray."""
    # prefix value -> first index where it appeared (-1 = before the array)
    first = {0: -1}
    prefix = best = 0
    for i, x in enumerate(nums):
        # 1 counts +1, 0 counts -1
        prefix += 1 if x else -1
        # same total again: the stretch in between is balanced
        if prefix in first:
            best = max(best, i - first[prefix])
        # keep only the first index (gives the longest stretch)
        else:
            first[prefix] = i
    return best


assert find_max_length([0, 1]) == 2
assert find_max_length([0, 1, 0]) == 2
assert find_max_length([0, 1, 1, 1, 1, 1, 0, 0, 0]) == 6
print(find_max_length([0, 1, 1, 1, 1, 1, 0, 0, 0]))

6


### 4.1.3 [974. Subarray Sums Divisible by K](https://leetcode.com/problems/subarray-sums-divisible-by-k/) — ★ · Medium

**Problem.** Count subarrays whose sum is divisible by `k`.  
**Signal.** Divisible ⇔ two prefixes have the same remainder mod `k`.  
*New here: Keying the map by remainder instead of the raw sum.*

**Fine print.** Ask before coding:
- Can values be negative? → Yes, so remainders must be normalised to be non-negative.
- Is `k ≥ 2`? → Yes.

> 💡 **Key insight.** `prefix[j] − prefix[i]` is divisible by `k` iff both prefixes have the same remainder mod `k`.

**Brute force →** Every subarray sum mod k → O(n²). Counting remainders → O(n).

**Steps**
1. Seed the remainder count with `{0: 1}`.
2. Update the running remainder `(prefix + x) % k`.
3. Add the count of that remainder to the answer, then increment it.

**Complexity**

- **Time:** O(n).
- **Space:** O(k) — at most k remainders.

⚠️ **Trap.** In other languages `%` can be negative; Python's `%` is already non-negative for positive `k`.

In [69]:
def subarrays_div_by_k(nums: list[int], k: int) -> int:
    """Count subarrays with sum % k == 0."""
    # remainder -> count; remainder 0 for the empty prefix
    seen = Counter({0: 1})
    prefix = total = 0
    for x in nums:
        prefix = (prefix + x) % k
        # pair with every earlier prefix of the same remainder
        total += seen[prefix]
        seen[prefix] += 1
    return total


assert subarrays_div_by_k([4, 5, 0, -2, -3, 1], 5) == 7
assert subarrays_div_by_k([5], 9) == 0
print(subarrays_div_by_k([4, 5, 0, -2, -3, 1], 5))

7


### 4.1.4 [523. Continuous Subarray Sum](https://leetcode.com/problems/continuous-subarray-sum/) — ★ · Medium

**Problem.** Is there a subarray of length ≥ 2 whose sum is a multiple of `k`?  
**Signal.** Same remainder twice, at least two indices apart.  
*New here: Combining remainder keys with a first-index map to enforce a minimum length.*

**Fine print.** Ask before coding:
- Length at least 2? → Yes, which is why the earliest index for each remainder is stored.
- Does 0 count as a multiple of `k`? → Yes.

> 💡 **Key insight.** A repeated remainder means a divisible subarray; storing the *first* index makes the length check as generous as possible.

**Brute force →** Every subarray of length ≥ 2 → O(n²). First-index remainders → O(n).

**Steps**
1. Seed `first = {0: -1}`.
2. Update the running remainder.
3. Seen before and `i − first ≥ 2` → `True`.
4. Not seen → store `i`; never overwrite an earlier index.

**Complexity**

- **Time:** O(n).
- **Space:** O(k).

In [70]:
def check_subarray_sum(nums: list[int], k: int) -> bool:
    """True if a length>=2 subarray sums to a multiple of k."""
    # remainder -> first index where it occurred
    first = {0: -1}
    prefix = 0
    for i, x in enumerate(nums):
        prefix = (prefix + x) % k
        if prefix in first:
            # same remainder at least two positions back
            if i - first[prefix] >= 2:
                return True
        else:
            first[prefix] = i
    return False


assert check_subarray_sum([23, 2, 4, 6, 7], 6)
assert check_subarray_sum([23, 2, 6, 4, 7], 6)
assert not check_subarray_sum([23, 2, 6, 4, 7], 13)
print(check_subarray_sum([23, 2, 4, 6, 7], 6))

True


## 4.2 Range queries and difference arrays

Without a hash map: precompute a prefix array (or a difference array) once, then answer
many queries in O(1). The 2D version adds inclusion–exclusion; the difference array runs
the idea in reverse for range *updates*.

### 4.2.1 [303. Range Sum Query - Immutable](https://leetcode.com/problems/range-sum-query-immutable/) — ★ · Easy

**Problem.** Answer many `sum(nums[left..right])` queries.  
**Signal.** Repeated range sums on a static array → prefix array.  
*New here: The plain prefix array with a leading 0.*

**Fine print.** Ask before coding:
- Can the array change between queries? → No; with updates use a Fenwick or segment tree.
- Are `left` and `right` inclusive? → Yes.

> 💡 **Key insight.** With `prefix[0] = 0`, every range sum is one subtraction: `prefix[right + 1] − prefix[left]`.

**Brute force →** Sum the range per query → O(n) per query. One O(n) build makes every query O(1).

**Steps**
1. Build `prefix` of length n + 1 starting with 0.
2. Each query returns `prefix[right + 1] − prefix[left]`.

**Complexity**

- **Time:** O(n) build, O(1) per query.
- **Space:** O(n) — the prefix array.

In [71]:
class NumArray:
    """Static range-sum structure."""

    def __init__(self, nums: list[int]) -> None:
        # prefix[i] = sum of the first i elements
        self.prefix = [0]
        for x in nums:
            self.prefix.append(self.prefix[-1] + x)

    def sum_range(self, left: int, right: int) -> int:
        """Sum of nums[left..right]."""
        # first right+1 elements minus first left elements
        return self.prefix[right + 1] - self.prefix[left]


arr = NumArray([-2, 0, 3, -5, 2, -1])
assert [arr.sum_range(0, 2), arr.sum_range(2, 5), arr.sum_range(0, 5)] == [1, -1, -3]
print(arr.sum_range(0, 5))

-3


### 4.2.2 [304. Range Sum Query 2D - Immutable](https://leetcode.com/problems/range-sum-query-2d-immutable/) — ★ · Medium

**Problem.** Many queries for the sum of a sub-rectangle.  
**Signal.** 2D prefix sums with inclusion–exclusion.  
*New here: Inclusion–exclusion in two dimensions.*

**Fine print.** Ask before coding:
- Will the matrix change? → No; with updates use a 2D BIT.
- Are corners inclusive? → Yes.

> 💡 **Key insight.** A rectangle = big top-left block − the block above − the block to the left + the corner block (it was subtracted twice).

**Brute force →** Sum each rectangle → O(mn) per query. A 2D prefix table makes each query O(1).

**Steps**
1. `P[r+1][c+1]` = sum of `(0,0)..(r,c)`, built from its up, left and up-left neighbours.
2. Query = `P[r2+1][c2+1] − P[r1][c2+1] − P[r2+1][c1] + P[r1][c1]`.

**Complexity**

- **Time:** O(mn) build, O(1) per query.
- **Space:** O(mn) — the table.

In [72]:
class NumMatrix:
    """Static 2D range-sum structure."""

    def __init__(self, matrix: list[list[int]]) -> None:
        rows, cols = len(matrix), len(matrix[0])
        # padded table: row 0 and column 0 stay zero
        self.p = [[0] * (cols + 1) for _ in range(rows + 1)]
        for r in range(rows):
            for c in range(cols):
                # cell + up block + left block - overlap counted twice
                self.p[r + 1][c + 1] = (
                    matrix[r][c] + self.p[r][c + 1] + self.p[r + 1][c] - self.p[r][c]
                )

    def sum_region(self, r1: int, c1: int, r2: int, c2: int) -> int:
        """Sum of matrix[r1..r2][c1..c2]."""
        p = self.p
        # whole - above - left + corner
        return p[r2 + 1][c2 + 1] - p[r1][c2 + 1] - p[r2 + 1][c1] + p[r1][c1]


m = NumMatrix(
    [
        [3, 0, 1, 4, 2],
        [5, 6, 3, 2, 1],
        [1, 2, 0, 1, 5],
        [4, 1, 0, 1, 7],
        [1, 0, 3, 0, 5],
    ]
)
assert m.sum_region(2, 1, 4, 3) == 8
assert m.sum_region(1, 1, 2, 2) == 11
assert m.sum_region(1, 2, 2, 4) == 12
print(m.sum_region(2, 1, 4, 3))

8


### 4.2.3 [724. Find Pivot Index](https://leetcode.com/problems/find-pivot-index/) — ★ · Easy

**Problem.** Leftmost index where the sum to the left equals the sum to the right.  
**Signal.** Left sum + right sum = total − nums[i] → one running sum.  
*New here: Getting the right-hand sum for free from the total.*

**Fine print.** Ask before coding:
- Several pivots? → Return the leftmost. None? → Return −1.
- At index 0, is the left sum 0? → Yes.

> 💡 **Key insight.** Right sum = `total − left_sum − nums[i]`, so one running left sum is enough.

**Brute force →** Sum both sides for every index → O(n²). Total + running sum → O(n).

**Steps**
1. Compute the total.
2. Walk left to right with `left_sum`.
3. If `left_sum == total − left_sum − nums[i]`, return `i`.
4. Add `nums[i]` to `left_sum` and continue.

**Complexity**

- **Time:** O(n) — two passes.
- **Space:** O(1).

In [73]:
def pivot_index(nums: list[int]) -> int:
    """Leftmost balance index, or -1."""
    total, left_sum = sum(nums), 0
    for i, x in enumerate(nums):
        # right side = total - left side - the pivot itself
        if left_sum == total - left_sum - x:
            return i
        left_sum += x
    return -1


assert pivot_index([1, 7, 3, 6, 5, 6]) == 3
assert pivot_index([1, 2, 3]) == -1
assert pivot_index([2, 1, -1]) == 0
print(pivot_index([1, 7, 3, 6, 5, 6]))

3


### 4.2.4 [1109. Corporate Flight Bookings](https://leetcode.com/problems/corporate-flight-bookings/) — ★ · Medium

**Problem.** Bookings `[first, last, seats]` add seats to a range of flights; return totals per flight.  
**Signal.** Many range-add updates, one final read → difference array.  
*New here: The difference array — prefix sums run in reverse.*

**Fine print.** Ask before coding:
- Are flight numbers 1-indexed and ranges inclusive? → Yes, both.
- How many bookings? → Up to 2·10⁴, so a difference array beats adding seats one by one.

> 💡 **Key insight.** Adding `v` to a range only changes the *differences* at its two ends: `+v` at the start, `−v` just after the end.

**Brute force →** Add seats to every flight in each range → O(n · b). Difference array → O(n + b).

**Steps**
1. `diff[first − 1] += seats`, `diff[last] −= seats` for each booking.
2. Prefix-sum the difference array.
3. The running sums are the per-flight totals.

**Complexity**

- **Time:** O(n + b) — O(1) per booking plus one pass.
- **Space:** O(n) — the difference array.

In [74]:
def corp_flight_bookings(bookings: list[list[int]], n: int) -> list[int]:
    """Seat totals per flight after range bookings."""
    diff = [0] * (n + 1)
    for first, last, seats in bookings:
        # the range starts adding here...
        diff[first - 1] += seats
        # ...and stops adding after `last`
        diff[last] -= seats
    # a prefix sum over the changes rebuilds the totals
    out, running = [], 0
    for i in range(n):
        running += diff[i]
        out.append(running)
    return out


assert corp_flight_bookings([[1, 2, 10], [2, 3, 20], [2, 5, 25]], 5) == [
    10,
    55,
    45,
    25,
    25,
]
assert corp_flight_bookings([[1, 2, 10], [2, 2, 15]], 2) == [10, 25]
print(corp_flight_bookings([[1, 2, 10], [2, 3, 20], [2, 5, 25]], 5))

[10, 55, 45, 25, 25]


### 4.2.5 [528. Random Pick with Weight](https://leetcode.com/problems/random-pick-with-weight/) — ★★ · Medium

**Problem.** Pick index `i` with probability `w[i] / sum(w)`.  
**Signal.** Sampling by weight → prefix sums + binary search on a uniform random number.  
*New here: Prefix sums as a cumulative distribution, searched with binary search.*

**Fine print.** Ask before coding:
- Are all weights positive? → Yes.
- How many `pick` calls? → Many (up to 10⁴), so precompute prefix sums and binary-search each pick.

> 💡 **Key insight.** Prefix sums cut `[0, total)` into segments of length `w[i]`; a uniform random point lands in segment `i` with probability `w[i] / total`.

**Brute force →** Expand index `i` into `w[i]` copies and pick uniformly → O(total) memory. Linear scan of prefixes → O(n) per pick. Binary search → O(log n).

**Steps**
1. Build the running prefix sums.
2. Draw an integer `target` in `[0, total)`.
3. Binary search for the first prefix greater than `target`.
4. Return that index.

**Complexity**

- **Time:** O(n) build, O(log n) per pick.
- **Space:** O(n).

In [75]:
class WeightedPicker:
    """Sample indices proportional to weights."""

    def __init__(self, w: list[int]) -> None:
        # prefix[i] = right edge of index i's segment
        self.prefix: list[int] = []
        running = 0
        for x in w:
            running += x
            self.prefix.append(running)

    def pick_index(self) -> int:
        """Random index drawn by weight."""
        target = random.randrange(self.prefix[-1])
        # first segment whose right edge is beyond target
        lo, hi = 0, len(self.prefix) - 1
        while lo < hi:
            mid = (lo + hi) // 2
            if self.prefix[mid] > target:
                hi = mid
            else:
                lo = mid + 1
        return lo


random.seed(0)
picker = WeightedPicker([1, 3])
freq = Counter(picker.pick_index() for _ in range(10_000))
assert 0.70 < freq[1] / 10_000 < 0.80
print(freq)

Counter({1: 7500, 0: 2500})


---
# 5) Matrix

**In plain English.** Matrix questions are index bookkeeping. Name the boundaries
(`top, bottom, left, right`), move them one at a time, and when you must update in place,
find a spare spot to remember things (the first row/column, or spare bits in each cell).

> **Analogy.** Peeling an onion: take the outer ring, then the next ring in, until nothing
> is left. Each ring is four straight walks.

**Recognition signals**

- "spiral", "layer by layer", "clockwise"
- "rotate 90°", "transpose", "in place"
- "set row and column to 0", "next state of every cell at once"
- "diagonal" (cells share `r − c` on a diagonal, `r + c` on an anti-diagonal)

**Common bugs**

- Single remaining row or column read twice in a spiral (guard with `top < bottom`,
  `left < right`).
- `[[0] * n] * m` creates m references to the *same* row.
- Updating cells in place while neighbours still need the old value (Game of Life).
- Overwriting the marker row/column before reading whether it had its own zero.

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Grid DFS / BFS | explores connected regions with a visited set; here the visiting order is fixed |
| Grid DP | each cell's value depends on neighbours via a recurrence; here it is a transform |
| 2D prefix sums | precomputes rectangle sums; here the matrix itself is rearranged |

**Useful identities.** Clockwise rotation = transpose + reverse each row. Counter-clockwise
= reverse each row + transpose. Diagonal key `r − c`; anti-diagonal key `r + c`.

**Learning path.** Spiral read → spiral write → rotate (transpose + reverse) → in-place
markers (Set Zeroes) → two states in one cell (Game of Life) → diagonal keys (Toeplitz,
Diagonal Traverse).

## 5.1 Traversal and in-place transforms

Two recurring tricks: shrinking four boundaries for layer-by-layer walks, and storing
extra state *inside* the matrix (first row/column as flags, bit 1 for the next state) to
stay O(1) space.

### 5.1.1 [54. Spiral Matrix](https://leetcode.com/problems/spiral-matrix/) — ★★★ · Medium

**Problem.** Return all elements in spiral order.  
**Signal.** Layer-by-layer walk with four shrinking boundaries.  
*New here: Four boundaries that shrink after each side is walked.*

**Fine print.** Ask before coding:
- Can the matrix be non-square or a single row or column? → Yes, which is why the boundary checks matter.
- Clockwise, starting at the top-left? → Yes.

> 💡 **Key insight.** Each layer is four straight walks (right, down, left, up); after a layer, every boundary moves one step inward.

**Brute force →** Simulate with direction vectors and a visited matrix → O(mn) time but O(mn) extra space. Boundaries need O(1).

**Steps**
1. Set `top, bottom, left, right`.
2. Read the top row left→right, then the right column top→bottom.
3. If a bottom row remains, read it right→left; if a left column remains, read it bottom→top.
4. Shrink all four boundaries; repeat while `top ≤ bottom` and `left ≤ right`.

**Complexity**

- **Time:** O(mn) — every cell is read once.
- **Space:** O(1) extra besides the output.

⚠️ **Trap.** Without the two "remains" checks, a single middle row or column is read twice.

In [76]:
def spiral_order(matrix: list[list[int]]) -> list[int]:
    """Elements in clockwise spiral order."""
    out = []
    top, bottom, left, right = 0, len(matrix) - 1, 0, len(matrix[0]) - 1
    while top <= bottom and left <= right:
        # top row, left to right
        out += [matrix[top][c] for c in range(left, right + 1)]
        # right column, downward
        out += [matrix[r][right] for r in range(top + 1, bottom + 1)]
        # bottom row, right to left (only if it is a different row)
        if top < bottom:
            out += [matrix[bottom][c] for c in range(right - 1, left - 1, -1)]
        # left column, upward (only if it is a different column)
        if left < right:
            out += [matrix[r][left] for r in range(bottom - 1, top, -1)]
        # peel the layer
        top, bottom, left, right = top + 1, bottom - 1, left + 1, right - 1
    return out


assert spiral_order([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [1, 2, 3, 6, 9, 8, 7, 4, 5]
assert spiral_order([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]) == [
    1,
    2,
    3,
    4,
    8,
    12,
    11,
    10,
    9,
    5,
    6,
    7,
]
assert spiral_order([[1], [2], [3]]) == [1, 2, 3]
print(spiral_order([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))

[1, 2, 3, 6, 9, 8, 7, 4, 5]


### 5.1.2 [59. Spiral Matrix II](https://leetcode.com/problems/spiral-matrix-ii/) — ★ · Medium

**Problem.** Fill an `n × n` matrix with `1..n²` in spiral order.  
**Signal.** Same boundary walk as Spiral Matrix, writing instead of reading.  
*New here: The same walk used for writing — confirms the template.*

**Fine print.** Ask before coding:
- Is the matrix always square? → Yes, `n × n`.
- Can `n` be 1? → Yes.

> 💡 **Key insight.** The visiting order is identical to Spiral Matrix; only the action changes from read to write.

**Brute force →** Direction vectors + "is this cell filled?" checks → also O(n²), just more state. Boundaries are simpler.

**Steps**
1. Create an n × n grid and a counter starting at 1.
2. Walk the four sides of the current layer, writing the counter.
3. Shrink the boundaries and repeat.

**Complexity**

- **Time:** O(n²) — each cell written once.
- **Space:** O(1) extra besides the output grid.

In [77]:
def generate_matrix(n: int) -> list[list[int]]:
    """n x n matrix filled in spiral order."""
    grid = [[0] * n for _ in range(n)]
    top, bottom, left, right = 0, n - 1, 0, n - 1
    value = 1
    while top <= bottom:
        # top row
        for c in range(left, right + 1):
            grid[top][c] = value
            value += 1
        # right column
        for r in range(top + 1, bottom + 1):
            grid[r][right] = value
            value += 1
        # bottom row (skip when the layer is a single row)
        for c in range(right - 1, left - 1, -1):
            if top < bottom:
                grid[bottom][c] = value
                value += 1
        # left column
        for r in range(bottom - 1, top, -1):
            grid[r][left] = value
            value += 1
        top, bottom, left, right = top + 1, bottom - 1, left + 1, right - 1
    return grid


assert generate_matrix(3) == [[1, 2, 3], [8, 9, 4], [7, 6, 5]]
assert generate_matrix(1) == [[1]]
print(generate_matrix(4))

[[1, 2, 3, 4], [12, 13, 14, 5], [11, 16, 15, 6], [10, 9, 8, 7]]


### 5.1.3 [48. Rotate Image](https://leetcode.com/problems/rotate-image/) — ★★★ · Medium

**Problem.** Rotate an `n × n` matrix 90° clockwise in place.  
**Signal.** Rotation = transpose + reverse each row.  
*New here: Decomposing a geometric transform into two simple in-place ones.*

**Fine print.** Ask before coding:
- Must it be in place with no second matrix? → Yes.
- Clockwise? → Yes; counter-clockwise swaps the order of transpose and reflect.

> 💡 **Key insight.** Clockwise rotation sends `(r, c)` to `(c, n−1−r)`: the transpose gives `(c, r)` and reversing each row turns `r` into `n−1−r`.

**Brute force →** Build a rotated copy → O(n²) extra space. Transpose + reverse uses O(1).

**Steps**
1. Transpose: swap `m[i][j]` and `m[j][i]` for `j > i`.
2. Reverse every row.
3. (Counter-clockwise: reverse rows first, then transpose.)

**Complexity**

- **Time:** O(n²) — each cell moves a constant number of times.
- **Space:** O(1).

In [78]:
def rotate_image(matrix: list[list[int]]) -> None:
    """Rotate square matrix clockwise in place."""
    n = len(matrix)
    # transpose: mirror across the main diagonal
    for i in range(n):
        for j in range(i + 1, n):
            matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]
    # mirror each row left-right
    for row in matrix:
        row.reverse()


m = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
rotate_image(m)
assert m == [[7, 4, 1], [8, 5, 2], [9, 6, 3]]
print(m)

[[7, 4, 1], [8, 5, 2], [9, 6, 3]]


### 5.1.4 [73. Set Matrix Zeroes](https://leetcode.com/problems/set-matrix-zeroes/) — ★★★ · Medium

**Problem.** If a cell is 0, set its whole row and column to 0, in place with O(1) space.  
**Signal.** Need markers without extra memory → use the first row and column as marker storage.  
*New here: Using part of the input as flag storage.*

**Fine print.** Ask before coding:
- Is O(1) extra space required? → Ideally; O(m + n) with two marker sets is the easy version.
- Must it be done in place? → Yes.

> 💡 **Key insight.** Row 0 and column 0 can hold the "zero this column / row" flags, as long as you first remember whether they had zeros of their own.

**Brute force →** Copy the matrix → O(mn) space. Row and column flag sets → O(m + n) space. First row/column as flags → O(1).

**Steps**
1. Record whether row 0 and column 0 originally contain a zero.
2. For every inner zero, set `m[r][0]` and `m[0][c]` to 0.
3. Zero every inner cell whose row or column flag is 0.
4. Finally zero row 0 and/or column 0 if their own flags were set.

**Complexity**

- **Time:** O(mn) — a constant number of passes.
- **Space:** O(1) — two booleans.

⚠️ **Trap.** Zeroing row 0 or column 0 before step 3 wipes out the flags.

In [79]:
def set_zeroes(matrix: list[list[int]]) -> None:
    """Zero rows/columns of every zero cell, O(1) space."""
    rows, cols = len(matrix), len(matrix[0])
    # remember row 0 / column 0 before they are used as flags
    first_row = any(matrix[0][c] == 0 for c in range(cols))
    first_col = any(matrix[r][0] == 0 for r in range(rows))
    # flag the row and column of every inner zero
    for r in range(1, rows):
        for c in range(1, cols):
            if matrix[r][c] == 0:
                matrix[r][0] = matrix[0][c] = 0
    # apply the flags to the inner cells
    for r in range(1, rows):
        for c in range(1, cols):
            if matrix[r][0] == 0 or matrix[0][c] == 0:
                matrix[r][c] = 0
    # finally handle the flag row and column themselves
    if first_row:
        matrix[0] = [0] * cols
    if first_col:
        for r in range(rows):
            matrix[r][0] = 0


m = [[0, 1, 2, 0], [3, 4, 5, 2], [1, 3, 1, 5]]
set_zeroes(m)
assert m == [[0, 0, 0, 0], [0, 4, 5, 0], [0, 3, 1, 0]]
m = [[1, 1, 1], [1, 0, 1], [1, 1, 1]]
set_zeroes(m)
assert m == [[1, 0, 1], [0, 0, 0], [1, 0, 1]]
print(m)

[[1, 0, 1], [0, 0, 0], [1, 0, 1]]


### 5.1.5 [289. Game of Life](https://leetcode.com/problems/game-of-life/) — ★★ · Medium

**Problem.** Compute the next state of Conway's Game of Life in place.  
**Signal.** Need old and new state at once → encode both in each cell's bits.  
*New here: Two states in one integer: bit 0 = now, bit 1 = next.*

**Fine print.** Ask before coding:
- Must it be in place? → Yes, so encode both the old and new state in each cell.
- Do cells beyond the border count? → No, they are dead.
- Follow-up: an infinite board? → Store only live cells in a set.

> 💡 **Key insight.** A cell's current state is `cell & 1` no matter what has been written into bit 1, so neighbours can still read the old generation.

**Brute force →** Copy the board and read from the copy → O(mn) space. Bit packing is O(1).

**Steps**
1. For each cell, count live neighbours using `& 1`.
2. If it lives next round (3 neighbours, or 2 and currently alive), set bit 1.
3. After all cells, shift every cell right by one to make "next" the new "now".

**Complexity**

- **Time:** O(mn) — 8 neighbours per cell.
- **Space:** O(1).

In [80]:
def game_of_life(board: list[list[int]]) -> None:
    """Advance the board one generation in place."""
    rows, cols = len(board), len(board[0])
    for r in range(rows):
        for c in range(cols):
            # bit 0 still holds every neighbour's current state
            live = sum(
                board[i][j] & 1
                for i in range(max(0, r - 1), min(rows, r + 2))
                for j in range(max(0, c - 1), min(cols, c + 2))
                if (i, j) != (r, c)
            )
            # survives or is born: store the next state in bit 1
            if live == 3 or (live == 2 and board[r][c] & 1):
                board[r][c] |= 2
    # next state becomes the current state
    for r in range(rows):
        for c in range(cols):
            board[r][c] >>= 1


b = [[0, 1, 0], [0, 0, 1], [1, 1, 1], [0, 0, 0]]
game_of_life(b)
assert b == [[0, 0, 0], [1, 0, 1], [0, 1, 1], [0, 1, 0]]
print(b)

[[0, 0, 0], [1, 0, 1], [0, 1, 1], [0, 1, 0]]


### 5.1.6 [766. Toeplitz Matrix](https://leetcode.com/problems/toeplitz-matrix/) — ★ · Easy

**Problem.** Is every top-left to bottom-right diagonal constant?  
**Signal.** Each cell must equal its upper-left neighbour.  
*New here: Turning "every diagonal is constant" into a local neighbour check.*

**Fine print.** Ask before coding:
- Can the matrix be non-square? → Yes.
- Follow-up: only one row fits in memory? → Compare each row with the previous row shifted by one.

> 💡 **Key insight.** A diagonal is constant iff every cell equals the one diagonally above-left of it.

**Brute force →** Walk each diagonal separately and check all values → same O(mn), more code. The local check is one line.

**Steps**
1. For every cell outside the first row and column, compare it with `m[r−1][c−1]`.
2. Any mismatch → `False`.

**Complexity**

- **Time:** O(mn).
- **Space:** O(1) — with streamed rows, only the previous row is needed.

In [81]:
def is_toeplitz_matrix(matrix: list[list[int]]) -> bool:
    """True if all diagonals are constant."""
    # every cell must equal its upper-left neighbour
    return all(
        matrix[r][c] == matrix[r - 1][c - 1]
        for r in range(1, len(matrix))
        for c in range(1, len(matrix[0]))
    )


assert is_toeplitz_matrix([[1, 2, 3, 4], [5, 1, 2, 3], [9, 5, 1, 2]])
assert not is_toeplitz_matrix([[1, 2], [2, 2]])
print("ok")

ok


### 5.1.7 [498. Diagonal Traverse](https://leetcode.com/problems/diagonal-traverse/) — ★ · Medium

**Problem.** Return matrix elements in zigzag diagonal order.  
**Signal.** Cells on one anti-diagonal share `r + c`.  
*New here: Grouping cells by a diagonal key.*

**Fine print.** Ask before coding:
- Which direction does the first diagonal go? → Up and to the right.
- Can the matrix be non-square? → Yes.

> 💡 **Key insight.** Every anti-diagonal is the set of cells with the same `r + c`; the zigzag just reverses every other group.

**Brute force →** Simulate the bouncing direction with edge cases at every border → error-prone. Grouping by `r + c` has no edge cases.

**Steps**
1. Scan rows top-down and append each value to group `r + c`.
2. For each `d` from 0 upward, emit group `d`.
3. Reverse even groups (they run bottom-left → top-right).

**Complexity**

- **Time:** O(mn).
- **Space:** O(mn) — the groups.

In [82]:
def find_diagonal_order(mat: list[list[int]]) -> list[int]:
    """Zigzag diagonal traversal."""
    # r + c identifies the anti-diagonal; rows scanned top-down
    groups: dict[int, list[int]] = defaultdict(list)
    for r, row in enumerate(mat):
        for c, value in enumerate(row):
            groups[r + c].append(value)
    out = []
    for d in range(len(mat) + len(mat[0]) - 1):
        # even diagonals travel upward, so reverse them
        out += groups[d][::-1] if d % 2 == 0 else groups[d]
    return out


assert find_diagonal_order([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [
    1,
    2,
    4,
    7,
    5,
    3,
    6,
    8,
    9,
]
assert find_diagonal_order([[1, 2], [3, 4]]) == [1, 2, 3, 4]
print(find_diagonal_order([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))

[1, 2, 4, 7, 5, 3, 6, 8, 9]


---
# 6) Strings

**In plain English.** Most string questions are careful simulation: follow the rules
exactly and get every edge case right on the first pass. A few hide a neat trick (a custom
sort order, a rotation identity).

> **Analogy.** Reading a form filled in by hand: skip the blank spaces, read the sign,
> read digits until something that isn't a digit, and stop. Following the steps in order
> is the whole job.

**Recognition signals**

- "parse", "convert", "valid …", "compress", "justify", "compare versions"
- a precise list of rules in the statement (atoi, text justification)
- "largest number formed by", "custom order" → sort with a comparator or a rank map

**Common bugs (list them before coding)**

- empty string, all spaces, trailing spaces
- sign handling and leading zeros (`"-042"`, `"01"`)
- overflow clamping (Python ints never overflow — clamp explicitly)
- building strings with `+=` in a loop (O(n²)) instead of a list + `"".join`
- multi-digit numbers inside a string (`"i12iz4n"`)

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Sliding window on strings | asks for the best substring; here you transform or validate the whole string |
| String DP | asks for counts/optima over subsequences or alignments; here it is direct simulation |
| Stack parsing | needed when the format nests (brackets); flat formats need only an index |

**Learning path.** Scans (common prefix, last word) → row simulation (Zigzag) → rule-by-rule
parser (atoi) → greedy packing (Text Justification) → in-place RLE → custom sort orders
(Largest Number, Custom Sort) → parsing digits inside text → zipped comparison → the
rotation identity (Repeated Substring).

## 6.1 Parsing and simulation

Before coding, write down the edge cases the statement implies — that list *is* most of
the solution here.

### 6.1.1 [14. Longest Common Prefix](https://leetcode.com/problems/longest-common-prefix/) — ★★ · Easy

**Problem.** Longest common prefix of an array of strings.  
**Signal.** Vertical scan, or compare only the lexicographic min and max.  
*New here: Reducing a many-string comparison to just two strings.*

**Fine print.** Ask before coding:
- Can the list be empty or contain an empty string? → Yes, then the answer is `""`.
- Only lowercase letters? → Yes.

> 💡 **Key insight.** In sorted order the first and last strings differ the most, so their common prefix is shared by everything between them.

**Brute force →** Compare every string character by character (vertical scan) → O(total chars), also fine. min/max is shorter to write.

**Steps**
1. Take `lo = min(strs)` and `hi = max(strs)`.
2. Advance `i` while `lo[i] == hi[i]`.
3. Return `lo[:i]`.

**Complexity**

- **Time:** O(total chars) — `min` and `max` compare strings.
- **Space:** O(1) extra.

In [83]:
def longest_common_prefix(strs: list[str]) -> str:
    """Longest prefix shared by all strings."""
    # the lexicographic extremes differ the most
    lo, hi = min(strs), max(strs)
    i = 0
    while i < len(lo) and lo[i] == hi[i]:
        i += 1
    return lo[:i]


assert longest_common_prefix(["flower", "flow", "flight"]) == "fl"
assert longest_common_prefix(["dog", "racecar", "car"]) == ""
print(longest_common_prefix(["flower", "flow", "flight"]))

fl


### 6.1.2 [58. Length of Last Word](https://leetcode.com/problems/length-of-last-word/) — ★★ · Easy

**Problem.** Length of the last word (words separated by spaces, possibly trailing spaces).  
**Signal.** Scan from the end.  
*New here: Scanning from the end to avoid processing the whole string.*

**Fine print.** Ask before coding:
- Can there be trailing or multiple spaces? → Yes.
- Is at least one word guaranteed? → Yes.

> 💡 **Key insight.** The last word is the first run of non-spaces you meet when reading backwards.

**Brute force →** `s.split()[-1]` → O(n) time and O(n) space. The backward scan uses O(1) and stops early.

**Steps**
1. From the end, skip spaces.
2. Mark the end of the word.
3. Move left until a space or the start; return the distance.

**Complexity**

- **Time:** O(n) worst case, usually only the last word is read.
- **Space:** O(1).

In [84]:
def length_of_last_word(s: str) -> int:
    """Length of the final word."""
    i = len(s) - 1
    # skip trailing spaces
    while s[i] == " ":
        i -= 1
    end = i
    # walk back over the word
    while i >= 0 and s[i] != " ":
        i -= 1
    return end - i


assert length_of_last_word("Hello World") == 5
assert length_of_last_word("   fly me   to   the moon  ") == 4
print(length_of_last_word("luffy is still joyboy"))

6


### 6.1.3 [6. Zigzag Conversion](https://leetcode.com/problems/zigzag-conversion/) — ★★ · Medium

**Problem.** Write `s` in a zigzag over `num_rows` rows, then read row by row.  
**Signal.** Simulate with a row index that bounces between 0 and `num_rows − 1`.  
*New here: Simulation with a bouncing index.*

**Fine print.** Ask before coding:
- Can `num_rows` be 1 or larger than `len(s)`? → Yes; then return `s` unchanged.

> 💡 **Key insight.** You never need the 2D picture — only which row each character lands in, and that row index bounces between the top and bottom.

**Brute force →** Fill an actual 2D grid and read it → O(n · rows) space. One string per row is O(n).

**Steps**
1. One accumulator per row; `row = 0`, `step = +1`.
2. Append each character to `rows[row]`.
3. Flip `step` at the top and bottom rows; move `row += step`.
4. Join the rows.

**Complexity**

- **Time:** O(n).
- **Space:** O(n).

⚠️ **Trap.** `num_rows == 1` never flips — return `s` directly.

In [85]:
def convert(s: str, num_rows: int) -> str:
    """Zigzag row-by-row reading of s."""
    if num_rows == 1:
        return s
    rows = [""] * num_rows
    row, step = 0, 1
    for ch in s:
        rows[row] += ch
        # bounce at the top and bottom rows
        if row == 0:
            step = 1
        elif row == num_rows - 1:
            step = -1
        row += step
    return "".join(rows)


assert convert("PAYPALISHIRING", 3) == "PAHNAPLSIIGYIR"
assert convert("PAYPALISHIRING", 4) == "PINALSIGYAHRPI"
assert convert("A", 1) == "A"
print(convert("PAYPALISHIRING", 3))

PAHNAPLSIIGYIR


### 6.1.4 [8. String to Integer (atoi)](https://leetcode.com/problems/string-to-integer-atoi/) — ★★ · Medium

**Problem.** Parse an int: skip leading spaces, optional sign, digits until a non-digit, clamp to 32-bit.  
**Signal.** Step-by-step parser; clamp at the end (Python has no overflow).  
*New here: A linear parser with clearly separated phases.*

**Fine print.** Ask before coding:
- Which characters can appear after the digits? → Anything; stop at the first non-digit.
- What happens on overflow? → Clamp to `[-2³¹, 2³¹ - 1]`.
- No digits at all? → Return 0.
- Are `+` and `-` together, or a space after the sign, allowed? → No, both give 0.

> 💡 **Key insight.** The format is four phases in a fixed order — spaces, sign, digits, stop — so one index moving forward handles it.

**Brute force →** Regex `^\s*[+-]?\d+` works but hides the logic. The index walk is explicit.

**Steps**
1. Skip leading spaces.
2. Read an optional `+` or `−`.
3. Accumulate digits until a non-digit.
4. Apply the sign and clamp to `[−2³¹, 2³¹ − 1]`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

⚠️ **Trap.** In fixed-width languages, check for overflow *before* `value * 10 + d`.

In [86]:
def my_atoi(s: str) -> int:
    """C-style atoi with 32-bit clamping."""
    i, n = 0, len(s)
    # leading spaces
    while i < n and s[i] == " ":
        i += 1
    # optional sign
    sign = 1
    if i < n and s[i] in "+-":
        sign = -1 if s[i] == "-" else 1
        i += 1
    # digits until the first non-digit
    value = 0
    while i < n and s[i].isdigit():
        value = value * 10 + int(s[i])
        i += 1
    # clamp to the 32-bit range
    return max(-(2**31), min(2**31 - 1, sign * value))


assert my_atoi("42") == 42
assert my_atoi("   -042") == -42
assert my_atoi("1337c0d3") == 1337
assert my_atoi("words 987") == 0
assert my_atoi("-91283472332") == -(2**31)
print(my_atoi("   -042"))

-42


### 6.1.5 [68. Text Justification](https://leetcode.com/problems/text-justification/) — ★★ · Hard

**Problem.** Fully justify words into lines of width `max_width`; last line left-justified.  
**Signal.** Greedy line packing + careful space distribution.  
*New here: Greedy packing plus `divmod` to spread a remainder fairly.*

**Fine print.** Ask before coding:
- Is every word no longer than `max_width`? → Yes.
- Where do extra spaces go when they do not split evenly? → To the leftmost gaps first.
- A line with one word, or the last line? → Left-justify and pad on the right.

> 💡 **Key insight.** Spaces per gap are `divmod(spaces, gaps)`: everyone gets the quotient and the leftmost `remainder` gaps get one extra.

**Brute force →** No smarter algorithm exists — the difficulty is entirely the rules: fit greedily, then distribute.

**Steps**
1. Pack words while `letters + len(word) + gaps ≤ max_width`.
2. Last line or single word → join with single spaces and pad right.
3. Otherwise `space, extra = divmod(max_width − letters, gaps)`.
4. The first `extra` gaps get `space + 1` spaces, the rest `space`.

**Complexity**

- **Time:** O(total chars).
- **Space:** O(total chars) — the output.

In [87]:
def full_justify(words: list[str], max_width: int) -> list[str]:
    """Fully justified lines of text."""
    lines, i = [], 0
    while i < len(words):
        # pack words[i:j] greedily; each gap needs at least one space
        j, letters = i, 0
        while j < len(words) and letters + len(words[j]) + (j - i) <= max_width:
            letters += len(words[j])
            j += 1
        gaps = j - i - 1
        # last line or one word: left-justify and pad right
        if j == len(words) or gaps == 0:
            line = " ".join(words[i:j])
            lines.append(line + " " * (max_width - len(line)))
        else:
            # equal share per gap; leftmost gaps take the remainder
            space, extra = divmod(max_width - letters, gaps)
            line = ""
            for k in range(i, j - 1):
                line += words[k] + " " * (space + (k - i < extra))
            lines.append(line + words[j - 1])
        i = j
    return lines


words = ["This", "is", "an", "example", "of", "text", "justification."]
assert full_justify(words, 16) == [
    "This    is    an",
    "example  of text",
    "justification.  ",
]
print(full_justify(["What", "must", "be", "acknowledgment", "shall", "be"], 16))

['What   must   be', 'acknowledgment  ', 'shall be        ']


### 6.1.6 [443. String Compression](https://leetcode.com/problems/string-compression/) — ★ · Medium

**Problem.** Compress runs in place (`aabccc` → `a2bc3`); return the new length.  
**Signal.** Run-length encoding with read and write pointers.  
*New here: The write pointer (from in-place arrays) applied to run-length encoding.*

**Fine print.** Ask before coding:
- Must it be in place with O(1) extra space? → Yes.
- How are counts above 9 written? → As several digits, e.g. `a12`.
- Is a count of 1 written? → No, only the letter.

> 💡 **Key insight.** The encoding of a run is never longer than the run itself, so the write pointer never overtakes the read pointer.

**Brute force →** Build a new string → O(n) space. Read/write pointers → O(1).

**Steps**
1. At `read`, find the end of the current run.
2. Write the character.
3. If the run is longer than 1, write each digit of its length.
4. Continue from the run's end; return `write`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [88]:
def compress(chars: list[str]) -> int:
    """In-place run-length encode; return new length."""
    write = read = 0
    while read < len(chars):
        # find the end of the run starting at read
        ch, start = chars[read], read
        while read < len(chars) and chars[read] == ch:
            read += 1
        chars[write] = ch
        write += 1
        # run length digits, only for runs longer than 1
        if read - start > 1:
            for digit in str(read - start):
                chars[write] = digit
                write += 1
    return write


c = list("aabbccc")
k = compress(c)
assert c[:k] == list("a2b2c3")
c = list("abbbbbbbbbbbb")
assert c[: compress(c)] == list("ab12")
print(k)

6


### 6.1.7 [179. Largest Number](https://leetcode.com/problems/largest-number/) — ★★ · Medium

**Problem.** Arrange non-negative integers to form the largest number (as a string).  
**Signal.** Custom ordering: `a` before `b` iff `a + b > b + a` as strings.  
*New here: Sorting with a custom comparator.*

**Fine print.** Ask before coding:
- Return a string? → Yes, because the number can be far too large for an int.
- All zeros? → Return `"0"`, not `"000"`.

> 💡 **Key insight.** For two pieces, the better order is whichever concatenation is larger; that pairwise rule is transitive, so sorting by it is globally optimal.

**Brute force →** Try every permutation → O(n!). Sorting with the comparator → O(n log n).

**Steps**
1. Convert numbers to strings.
2. Sort with comparator `a + b > b + a → a first`.
3. Join; if the first piece is `"0"`, the answer is `"0"`.

**Complexity**

- **Time:** O(n log n · k) — each comparison concatenates two k-digit strings.
- **Space:** O(n · k).

⚠️ **Trap.** `[0, 0]` must return `"0"`, not `"00"`.

In [89]:
def largest_number(nums: list[int]) -> str:
    """Largest concatenation of the numbers."""
    strs = [str(x) for x in nums]
    # a goes first when "a then b" beats "b then a"
    strs.sort(key=cmp_to_key(lambda a, b: -1 if a + b > b + a else 1))
    # all zeros collapse to a single "0"
    return "0" if strs[0] == "0" else "".join(strs)


assert largest_number([10, 2]) == "210"
assert largest_number([3, 30, 34, 5, 9]) == "9534330"
assert largest_number([0, 0]) == "0"
print(largest_number([3, 30, 34, 5, 9]))

9534330


### 6.1.8 [791. Custom Sort String](https://leetcode.com/problems/custom-sort-string/) — ★ · Medium

**Problem.** Permute `s` so its characters follow the order given in `order`.  
**Signal.** Counting sort with a custom key order.  
*New here: Counting sort driven by an external order.*

**Fine print.** Ask before coding:
- Are characters in `order` unique? → Yes.
- Characters of `s` that are not in `order`? → Place them anywhere.

> 💡 **Key insight.** Count `s` once; then emitting characters in `order`'s sequence *is* the sort.

**Brute force →** `sorted(s, key=rank)` → O(n log n). Counting → O(n).

**Steps**
1. Count the characters of `s`.
2. For each character in `order`, emit it `count` times and remove it.
3. Emit the leftover characters in any order.

**Complexity**

- **Time:** O(|s| + |order|).
- **Space:** O(alphabet).

In [90]:
def custom_sort_string(order: str, s: str) -> str:
    """s rearranged to respect order."""
    count = Counter(s)
    # characters named in order, in that sequence
    out = [ch * count.pop(ch, 0) for ch in order]
    # everything not mentioned in order, anywhere after
    out += [ch * n for ch, n in count.items()]
    return "".join(out)


assert custom_sort_string("cba", "abcd") == "cbad"
assert custom_sort_string("bcafg", "abcd") == "bcad"
print(custom_sort_string("cba", "abcd"))

cbad


### 6.1.9 [408. Valid Word Abbreviation](https://leetcode.com/problems/valid-word-abbreviation/) — ★ · Easy · Premium

**Problem.** Does `abbr` (numbers replace skipped letters, no leading zeros) match `word`?  
**Signal.** Two pointers; parse multi-digit numbers in the abbreviation.  
*New here: Parsing a multi-digit number in the middle of text.*

**Fine print.** Ask before coding:
- Are leading zeros in numbers allowed? → No, `"a01"` is invalid.
- Is `"0"` a valid number? → No.
- Are adjacent numbers allowed? → No, a number is always a maximal run of digits.

> 💡 **Key insight.** A number in `abbr` is just a jump of the word pointer; letters must match one for one.

**Brute force →** Expand the abbreviation into wildcards and compare → same O(n), more memory. Two pointers do it in place.

**Steps**
1. Pointer `i` in `word`, `j` in `abbr`.
2. Digit: reject a leading `0`; read the whole number and jump `i` by it.
3. Letter: it must equal `word[i]`; advance both.
4. Valid iff both pointers end exactly at their ends.

**Complexity**

- **Time:** O(|abbr|).
- **Space:** O(1).

In [91]:
def valid_word_abbreviation(word: str, abbr: str) -> bool:
    """True if abbr abbreviates word."""
    i = j = 0
    while j < len(abbr):
        if abbr[j].isdigit():
            # numbers may not have leading zeros
            if abbr[j] == "0":
                return False
            # read the full number and skip that many letters
            num = 0
            while j < len(abbr) and abbr[j].isdigit():
                num = num * 10 + int(abbr[j])
                j += 1
            i += num
        else:
            # a letter must match the word letter exactly
            if i >= len(word) or word[i] != abbr[j]:
                return False
            i += 1
            j += 1
    return i == len(word)


assert valid_word_abbreviation("internationalization", "i12iz4n")
assert not valid_word_abbreviation("apple", "a2e")
assert not valid_word_abbreviation("a", "01")
print("ok")

ok


### 6.1.10 [165. Compare Version Numbers](https://leetcode.com/problems/compare-version-numbers/) — ★ · Medium

**Problem.** Compare version strings like `1.01` vs `1.001.0`; return −1, 0 or 1.  
**Signal.** Split on dots, compare integer revisions, pad the shorter with zeros.  
*New here: `zip_longest` with a fill value for unequal lengths.*

**Fine print.** Ask before coding:
- Are leading zeros ignored? → Yes, `"01" == "1"`.
- Is a missing revision treated as 0? → Yes, `"1.0" == "1"`.

> 💡 **Key insight.** Missing revisions behave like 0, and `int()` removes leading zeros, so a padded integer comparison is exact.

**Brute force →** Compare strings directly → wrong (`"10" < "2"`). Integer revisions fix it.

**Steps**
1. Split both versions on `.` and convert each part to `int`.
2. Walk them in parallel, padding the shorter with 0.
3. Return at the first difference; equal throughout → 0.

**Complexity**

- **Time:** O(m + n).
- **Space:** O(m + n) for the split parts.

In [92]:
def compare_version(version1: str, version2: str) -> int:
    """Sign of version1 - version2."""
    # int() drops leading zeros
    a = map(int, version1.split("."))
    b = map(int, version2.split("."))
    # missing revisions count as 0
    for x, y in zip_longest(a, b, fillvalue=0):
        if x != y:
            return 1 if x > y else -1
    return 0


assert compare_version("1.2", "1.10") == -1
assert compare_version("1.01", "1.001") == 0
assert compare_version("1.0", "1.0.0.0") == 0
print(compare_version("1.2", "1.10"))

-1


### 6.1.11 [459. Repeated Substring Pattern](https://leetcode.com/problems/repeated-substring-pattern/) — ★ · Easy

**Problem.** Can `s` be built by repeating one of its substrings?  
**Signal.** Rotation trick: `s` is periodic iff it appears inside `(s + s)[1:-1]`.  
*New here: A string identity that replaces a search over all periods.*

**Fine print.** Ask before coding:
- Must the substring repeat at least twice? → Yes, so `s` itself does not count.
- Only lowercase letters? → Yes.

> 💡 **Key insight.** If `s` is made of a repeated block, rotating it by one block gives `s` back, so `s` appears inside `s + s` somewhere other than the two trivial positions.

**Brute force →** For each divisor `d` of n, test whether `s[:d] * (n // d) == s` → O(n · divisors). The rotation check is one substring search.

**Steps**
1. Build `s + s`.
2. Remove the first and last characters (kills the matches at 0 and n).
3. `s` is periodic iff it still occurs inside.

**Complexity**

- **Time:** O(n) with a linear-time substring search (KMP).
- **Space:** O(n) — the doubled string.

In [93]:
def repeated_substring_pattern(s: str) -> bool:
    """True if s is a repetition of a shorter substring."""
    # a non-trivial rotation equal to s exists iff s is periodic
    return s in (s + s)[1:-1]


assert repeated_substring_pattern("abab")
assert not repeated_substring_pattern("aba")
assert repeated_substring_pattern("abcabcabcabc")
print(repeated_substring_pattern("abab"))

True


---
# Summary

| Pattern | Recognise it by | Core move | Cost |
|---|---|---|---|
| Hash map / set | pair sum, duplicates, grouping, frequency | store what you'll look up later | O(n) |
| Canonical key | anagrams, isomorphism | sorted string / count tuple / first-index pattern | O(n·k) |
| Write pointer | in-place remove/dedupe/partition | `nums[write] = x` when kept | O(n), O(1) |
| Opposite pointers | sorted pair/triple sum, palindrome, container | move the side that cannot be in the answer | O(n) |
| Sliding window | longest/shortest contiguous block, positive values | expand right, shrink left while invalid | O(n) |
| Exactly-k trick | "exactly k distinct" | `at_most(k) − at_most(k−1)` | O(n) |
| Monotonic deque | window max/min | pop dominated elements from the back | O(n) |
| Prefix sum + map | subarray sum = k / divisible, negatives allowed | look up `prefix − k` | O(n) |
| Difference array | many range updates, one read | `+v` at start, `−v` after end | O(n + q) |
| Matrix boundaries | spiral, rotate, in-place marks | four boundaries; transpose + reverse; reuse row 0 | O(mn) |